In [1]:
from gettext import install
import pandas as pd
import numpy as np

from math import sqrt
from pathlib import Path

import warnings
from enum import Enum

from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import json
from pathlib import Path

from sklearn.model_selection import train_test_split


from pandas.core.interchange import dataframe

import matplotlib.pyplot as plt
from statsmodels.iolib import summary

from numpy.linalg import lstsq
from scipy.stats import kruskal
import scikit_posthocs as sp
import statsmodels.formula.api as smf
import seaborn as sns
import xgboost
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score


## File paths

In [2]:
input_path = Path("data/input/06_input")
output_path = Path("data/output/06_output")

## Parameters

In [3]:
anchor_correlated_outcome_variables = [
    # pain_right — |rho| >= 0.20 threshold applied
    "V06KOOSKPR",  # KOOS Pain right knee — preferred over WOMAC (contains WOMAC, 0-100 scale)
    "V06P7RKRCV",  # Right knee pain: severity, past 7 days, rated on scale of 0-10
    "V06ICPTSKR",  # ICOAP Right knee: Intermittent and Constant Pain Total Score, 0-100
    "V06KGLRS",    # Global knee rating — patient impression, distinct construct

    # pain_left — |rho| >= 0.20 threshold applied
    "V06KOOSKPL",  # KOOS Pain left knee
    "V06P7LKRCV",  # Left knee pain: severity, past 7 days, rated on scale of 0-10
    "V06ICPTSKL",  # ICOAP Left knee: Intermittent and Constant Pain Total Score

    # sleep — |rho| >= 0.10 threshold applied / "V06WPLKN3" > 0.20
    "V06WPLKN3",   # Left knee pain in bed
    "V06WPRKN3",   # Right knee pain in bed
    "V06CPRKN2",   # Right knee constant pain affecting sleep

    # function — |rho| >= 0.10 threshold applied
    "V0620MPACE",  # 20m walk pace (m/s) — represents full 20m walk cluster
    "V06CSPACE",   # Repeated chair stand: pace in stands/sec
    "V06STEPST1",  # 20-meter walk: trial 1 number of steps
    "V06400MTIM",  # 400-meter walk: total time at 400m or at stop
    "V06400MTR",   # 400-meter walk: total meters walked
    "V06CSTSGL",   # Single chair stand

    # coping_strategies — |rho| >= 0.10 threshold applied
    "V06CSQCAT",   # Coping Strategies Questionnaire Score, Catastrophizing

    # adl — |rho| >= 0.10 threshold applied
    "V06LLD6B",    # LLDI: Extent feel limited in taking part in active recreation
    "V06LLD6A",    # LLDI: How often take part in active recreation
    "V06LLDILST",  # LLDI: Limitation Dimension, Total Score (calculated)
    "V06LLD10B",   # LLDI: Extent feel limited in taking part in regular fitness program
    "V06ADL7A",    # ADL: Use equipment or device when getting in or out of bed

    # self_reported_function_right — |rho| >= 0.20 threshold applied
    "V06DIRKN1",   # Right knee difficulty: down stairs, last 7 days
    "V06DIRKN3",   # Right knee difficulty: stand from sitting, last 7 days
    "V06DIRKN7",   # Right knee difficulty: in car/out of car, last 7 days
    "V06DIRKN2",   # Right knee difficulty: up stairs, last 7 days
    "V06DIRKN8",   # Right knee difficulty: shopping, last 7 days
    "V06DIRKN5",   # Right knee difficulty: bending, last 7 days
    "V06DIRKN6",   # Right knee difficulty: walking, last 7 days
    "V06DIRKN15",  # Right knee difficulty: on/off toilet, last 7 days
    "V06DIRKN16",  # Right knee difficulty: heavy chores, last 7 days
    "V06DIRKN4",   # Right knee difficulty: standing, last 7 days
    "V06DIRKN9",   # Right knee difficulty: socks on, last 7 days
    "V06DIRKN17",  # Right knee difficulty: light chores, last 7 days
    "V06DIRKN13",  # Right knee difficulty: get in/out of bathtub, last 7 days

    # self_reported_function_left — |rho| >= 0.20 threshold applied
    "V06DILKN1",   # Left knee difficulty: down stairs, last 7 days
    "V06DILKN7",   # Left knee difficulty: in car/out of car, last 7 days
    "V06DILKN3",   # Left knee difficulty: stand from sitting, last 7 days
    "V06DILKN6",   # Left knee difficulty: walking, last 7 days
    "V06DILKN8",   # Left knee difficulty: shopping, last 7 days
    "V06DILKN2",   # Left knee difficulty: up stairs, last 7 days
    "V06DILKN4",   # Left knee difficulty: standing, last 7 days
    "V06DILKN5",   # Left knee difficulty: bending, last 7 days
    "V06DILKN16",  # Left knee difficulty: heavy chores, last 7 days
    "V06DILKN13",  # Left knee difficulty: get in/out of bathtub, last 7 days
    "V06DILKN10",  # Left knee difficulty: get out of bed, last 7 days
    "V06DILKN15",  # Left knee difficulty: on/off toilet, last 7 days
    "V06DILKN9",   # Left knee difficulty: socks on, last 7 days
    "V06DILKN17",  # Left knee difficulty: light chores, last 7 days
    "V06DILKN12",  # Left knee difficulty: lying down, last 7 days

    # self_reported_function_bilateral — |rho| >= 0.20 threshold applied
    "V06KOOSFX2",  # Either knee difficulty: running, last 7 days
    "V06KOOSFX4",  # Either knee difficulty: twisting/pivoting on injured knee, last 7 days
    "V06KOOSFX1",  # Either knee difficulty: squatting, last 7 days
    "V06KOOSFX5",  # Either knee difficulty: kneeling, last 7 days
    # activity_pase — |rho| >= 0.10 threshold applied
    "V06HOUACT6",  # Household activities: caring for another person, past 7 days
    "V06PASE6HR",  # Leisure activities: muscle strength/endurance, hours per day, past 7 days
]

In [4]:
PREDICTOR_COLUMNS = [
    "V06AACNT",
    "V06AALTMNT",
    "V06AAMDMNT",
    "V06AAMVMNT",
    "V06AAVMNT",
    "V06AAMVBMT",
    "V06AAVBMT",
    "V06ANVDAYS",
    "V06AACSM03",
    "V06ADHHS8",
    "V06ADHHSD8",
    "wake_minute_mean",
    "wake_minute_sd",
    "V06CESD11",
    "sleep_minute_mean",
    "sleep_minute_sd",
    "wear_duration_mean",
    "valid_days",
    "weekly_guideline_gap_minutes",
    "mean_sedentary_bout_count",
    "mean_sedentary_bout_mean_duration",
    "mean_sedentary_bout_max_duration",
    "mean_sedentary_bout_total_minutes",
    "mean_light_bout_count",
    "mean_light_bout_mean_duration",
    "mean_light_bout_max_duration",
    "mean_light_bout_total_minutes",
    "mean_moderate_bout_count",
    "mean_moderate_bout_mean_duration",
    "mean_moderate_bout_max_duration",
    "mean_moderate_bout_total_minutes",
    "mean_vigorous_bout_count",
    "mean_vigorous_bout_mean_duration",
    "mean_vigorous_bout_max_duration",
    "mean_vigorous_bout_total_minutes",
    "mean_mvpa_bout_count",
    "mean_mvpa_bout_mean_duration",
    "mean_mvpa_bout_max_duration",
    "mean_mvpa_bout_total_minutes",
    "mean_active_bout_count",
    "mean_active_bout_mean_duration",
    "mean_active_bout_max_duration",
    "mean_active_bout_total_minutes",
]

In [5]:
HARMONIC_PREDICTOR_COLUMNS = [
'mesor', 'amplitude', 'acrophase', 'intradaily_variability', 'interdaily_stability', 'mesor_mean_curve_weekday', 'mesor_mean_curve_weekend', 'amplitude_mean_curve_weekday', 'amplitude_mean_curve_weekend', 'acrophase_mean_curve_weekday', 'acrophase_mean_curve_weekend', 'iv_weekday', 'iv_weekend', 'iv_contrast']

In [6]:
FINAL_PREDICTOR_COLUMNS = [
    # Sleep / wear
    "wake_minute_mean",                # Mean wake time across valid days
    "wake_minute_sd",                  # Standard deviation of wake time
    "wear_duration_mean",              # Mean daily wear duration in minutes

    # Bout structure: sedentary
    "mean_sedentary_bout_count",
    "mean_sedentary_bout_mean_duration",
    "mean_sedentary_bout_max_duration",
    "mean_sedentary_bout_total_minutes",

    # Bout structure: light
    "mean_light_bout_max_duration",

    # Bout structure: MVPA
    "mean_mvpa_bout_count",
    "mean_mvpa_bout_mean_duration",

    # Bout structure: active
    "mean_active_bout_mean_duration",
    "mean_active_bout_max_duration",
    "mean_active_bout_total_minutes",

    # Harmonic features: overall
    "acrophase",                       # Overall acrophase retained — not involved in any redundant pair
    "interdaily_stability",            # IS retained as overall only — day-split not meaningful

    # Harmonic features: day-type specific
    "mesor_mean_curve_weekday",        # Mean activity level on weekdays
    "mesor_mean_curve_weekend",        # Mean activity level on weekends
    "amplitude_mean_curve_weekday",    # Peak-to-mean swing on weekdays
    "amplitude_mean_curve_weekend",    # Peak-to-mean swing on weekends
    "acrophase_mean_curve_weekday",    # Timing of activity peak on weekdays
    "acrophase_mean_curve_weekend",    # Timing of activity peak on weekends

    # IV: day-type specific
    "iv_weekday",                      # Within-day fragmentation on weekdays
    "iv_weekend",                      # Within-day fragmentation on weekends
    "iv_contrast",                     # iv_weekday - iv_weekend: work-related fragmentation burden
]

## Read data

In [7]:
summary_metrics_06 = pd.read_csv(output_path / "summary_metrics_06.csv")

In [8]:
summary_metrics_06 = pd.read_csv(
    output_path / "summary_metrics_06.csv",
    sep="|",
    na_values=[" ", ""],
)

# 1. Prediction models without harmonic features

## 1.1. Linear Regression

In [9]:
def predict_outcome(
        dataframe: pd.DataFrame,
        predictor_columns: list[str],
        outcome_column: str,
) -> None:
    """
    Fits a multivariate OLS linear regression model to predict a single
    outcome variable from a given set of predictors and prints a full
    model summary.

    :param dataframe: DataFrame containing all predictor and outcome columns.
    :param predictor_columns: List of column names to use as predictors.
    :param outcome_column: Column name of the outcome variable to predict.
    :return: None
    :raises TypeError: If any predictor column cannot be cast to float64.
    """
    clean_dataframe = (
        dataframe[predictor_columns + [outcome_column]]
        .replace(r'^\s*$', np.nan, regex=True)
        .dropna()
    )

    try:
        predictor_matrix = clean_dataframe[predictor_columns].to_numpy(dtype=np.float64)
    except (ValueError, TypeError) as error:
        non_numeric_columns = (
            clean_dataframe[predictor_columns]
            .select_dtypes(exclude="number")
            .columns
            .tolist()
        )
        raise TypeError(
            f"Could not cast predictor columns to float64. "
            f"Non-numeric columns detected: {non_numeric_columns}"
        ) from error

    outcome_vector = clean_dataframe[outcome_column].to_numpy(dtype=np.float64)

    predictor_matrix_train, predictor_matrix_test, outcome_train, outcome_test = train_test_split(
        predictor_matrix,
        outcome_vector,
        test_size=0.2,
        random_state=42,
    )

    model = LinearRegression()
    model.fit(predictor_matrix_train, outcome_train)

    outcome_train_predicted = model.predict(predictor_matrix_train)
    outcome_test_predicted = model.predict(predictor_matrix_test)
    residuals = outcome_train - outcome_train_predicted

    number_of_samples, number_of_predictors = predictor_matrix_train.shape
    degrees_of_freedom = number_of_samples - number_of_predictors - 1

    sum_of_squares_residual = np.sum(residuals ** 2)
    sum_of_squares_total = np.sum((outcome_train - outcome_train.mean()) ** 2)
    r_squared = 1 - sum_of_squares_residual / sum_of_squares_total
    adjusted_r_squared = 1 - (1 - r_squared) * (number_of_samples - 1) / degrees_of_freedom
    f_statistic = (r_squared / number_of_predictors) / ((1 - r_squared) / degrees_of_freedom)
    f_p_value = stats.f.sf(f_statistic, number_of_predictors, degrees_of_freedom)

    train_root_mean_squared_error = sqrt(mean_squared_error(outcome_train, outcome_train_predicted))
    test_root_mean_squared_error = sqrt(mean_squared_error(outcome_test, outcome_test_predicted))
    test_r_squared = r2_score(outcome_test, outcome_test_predicted)

    shapiro_statistic, shapiro_p_value = stats.shapiro(residuals)
    durbin_watson_statistic = np.sum(np.diff(residuals) ** 2) / sum_of_squares_residual

    # ── Print results ──────────────────────────────────────────────────────────

    print("=" * 60)
    print(f"MODEL: Predicting '{outcome_column}'")
    print(f"Predictors: {predictor_columns}")
    print("=" * 60)

    print("\n--- COEFFICIENTS ---")
    print(f"  Intercept: {model.intercept_:.4f}")
    for predictor_column, coefficient in zip(predictor_columns, model.coef_):
        print(f"  {predictor_column}: {coefficient:.4f}")

    print("\n--- MODEL FIT ---")
    print(f"  R²:          {r_squared:.4f}")
    print(f"  Adjusted R²: {adjusted_r_squared:.4f}")
    print(f"  F-statistic: {f_statistic:.4f}  (p={f_p_value:.4g})")
    print(f"  n={number_of_samples}  predictors={number_of_predictors}  df={degrees_of_freedom}")

    print("\n--- TRAIN / TEST PERFORMANCE (80/20 split) ---")
    print(f"  Train — R²: {r_squared:.4f}  RMSE: {train_root_mean_squared_error:.4f}")
    print(f"  Test  — R²: {test_r_squared:.4f}  RMSE: {test_root_mean_squared_error:.4f}")
    if r_squared - test_r_squared > 0.1:
        print(f"  ⚠ R² gap of {r_squared - test_r_squared:.3f} may indicate overfitting.")

    print("\n--- RESIDUAL DIAGNOSTICS ---")
    print(f"  Durbin-Watson: {durbin_watson_statistic:.4f}  (2 = no autocorrelation)")
    print(f"  Shapiro-Wilk:  W={shapiro_statistic:.4f}  p={shapiro_p_value:.4f}", end="  ")
    print("✓ approx. normal" if shapiro_p_value > 0.05 else "⚠ may not be normal")


predictor_columns = [
    # Derived sleep metrics (accelerometry-based)
    "wake_minute_mean",
    "wake_minute_sd",
    "wear_duration_mean",

    # Bout structure: sedentary
    "mean_sedentary_bout_count",
    "mean_sedentary_bout_mean_duration",
    "mean_sedentary_bout_max_duration",
    "mean_sedentary_bout_total_minutes",

    # Bout structure: light
    "mean_light_bout_count",
    "mean_light_bout_mean_duration",
    "mean_light_bout_max_duration",

    # Bout structure: MVPA
    "mean_mvpa_bout_count",
    "mean_mvpa_bout_mean_duration",
    "mean_mvpa_bout_max_duration",

    # Bout structure: active (all non-sedentary)
    "mean_active_bout_count",
    "mean_active_bout_mean_duration",
    "mean_active_bout_max_duration",
    "mean_active_bout_total_minutes",
]

summary_metrics_06["koos_pain_bilateral"] = summary_metrics_06[["V06KOOSKPR", "V06KOOSKPL"]].min(axis=1)

outcome_column = "koos_pain_bilateral"


/var/folders/tt/rfxy3hb13_1c3g0mkr2qww180000gn/T/ipykernel_6039/1139392432.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  summary_metrics_06["koos_pain_bilateral"] = summary_metrics_06[["V06KOOSKPR", "V06KOOSKPL"]].min(axis=1)


In [10]:
predict_outcome(
    dataframe=summary_metrics_06,
    predictor_columns=predictor_columns,
    outcome_column=outcome_column,
)

MODEL: Predicting 'koos_pain_bilateral'
Predictors: ['wake_minute_mean', 'wake_minute_sd', 'wear_duration_mean', 'mean_sedentary_bout_count', 'mean_sedentary_bout_mean_duration', 'mean_sedentary_bout_max_duration', 'mean_sedentary_bout_total_minutes', 'mean_light_bout_count', 'mean_light_bout_mean_duration', 'mean_light_bout_max_duration', 'mean_mvpa_bout_count', 'mean_mvpa_bout_mean_duration', 'mean_mvpa_bout_max_duration', 'mean_active_bout_count', 'mean_active_bout_mean_duration', 'mean_active_bout_max_duration', 'mean_active_bout_total_minutes']

--- COEFFICIENTS ---
  Intercept: 65.2527
  wake_minute_mean: 0.0049
  wake_minute_sd: -0.0035
  wear_duration_mean: -0.0019
  mean_sedentary_bout_count: -3.3577
  mean_sedentary_bout_mean_duration: -0.0771
  mean_sedentary_bout_max_duration: -0.0199
  mean_sedentary_bout_total_minutes: 0.0356
  mean_light_bout_count: -1.5880
  mean_light_bout_mean_duration: -2.4721
  mean_light_bout_max_duration: -0.0844
  mean_mvpa_bout_count: 1.2889
  m

## 1.2. Lasso, Elastic Net, Random Forest

In [11]:
RANDOM_STATE = 42
CROSS_VALIDATION_FOLDS = 10
TEST_SET_FRACTION = 0.20

COVARIATE_COLUMNS = ["V06AGE", "V06BMI", "kl_grade_index_knee", "V06COMORB"]

KL_GRADE_RIGHT_COLUMN = "V06XRKL_Right"
KL_GRADE_LEFT_COLUMN = "V06XRKL_Left"

In [12]:
def prepare_modelling_dataset(
    dataframe: pd.DataFrame,
    outcome_column: str,
    predictor_columns: list[str],
    covariate_columns: list[str] = COVARIATE_COLUMNS,
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    Prepare the modelling dataset by dropping rows with missing values
    in any covariate, predictor, or outcome column, then splitting into
    an 80 % training set and a 20 % held-out test set.

    Stratification for the split is performed on KL grade (binned into
    three groups: 0–1, 2–3, 4) to preserve the severity distribution
    across both splits.

    :param dataframe: Merged DataFrame containing covariates, predictors,
        and outcome variables (one row per patient).
    :param outcome_column: Name of the column to use as the prediction
        target.
    :param predictor_columns: List of activity predictor column names.
    :param covariate_columns: List of covariate column names that are
        always included as a fixed block.
    :returns: Tuple of
        (training_predictors, training_outcome,
         test_predictors, test_outcome)
        where predictors include both covariates and activity columns.
    :raises ValueError: If fewer than 100 complete cases remain after
        dropping missing values, which would make modelling unreliable.
    """
    all_required_columns = covariate_columns + predictor_columns + [outcome_column]
    complete_cases = dataframe[all_required_columns].dropna()

    complete_case_count = len(complete_cases)
    total_count = len(dataframe)
    print(
        f"Complete cases for '{outcome_column}':\n"
        f"  Total patients  : {total_count}\n"
        f"  Complete cases  : {complete_case_count} "
        f"({100 * complete_case_count / total_count:.1f} %)\n"
        f"  Dropped (missing): {total_count - complete_case_count}"
    )

    if complete_case_count < 100:
        raise ValueError(
            f"Only {complete_case_count} complete cases remain for "
            f"'{outcome_column}'. Modelling would be unreliable. "
            f"Consider imputation or relaxing inclusion criteria."
        )

    outcome_series = complete_cases[outcome_column]
    all_predictor_columns = covariate_columns + predictor_columns
    predictor_dataframe = complete_cases[all_predictor_columns]

    # Stratify split on KL grade groups to preserve severity distribution
    kl_grade_bins = pd.cut(
        complete_cases["kl_grade_index_knee"],
        bins=[-0.5, 1.5, 3.5, 4.5],
        labels=["kl_0_1", "kl_2_3", "kl_4"],
    )

    (
        training_predictors,
        test_predictors,
        training_outcome,
        test_outcome,
    ) = train_test_split(
        predictor_dataframe,
        outcome_series,
        test_size=TEST_SET_FRACTION,
        random_state=RANDOM_STATE,
        stratify=kl_grade_bins,
    )

    print(
        f"Train/test split (stratified by KL grade):\n"
        f"  Training set : {len(training_predictors)} patients\n"
        f"  Test set     : {len(test_predictors)} patients"
    )

    return training_predictors, training_outcome, test_predictors, test_outcome


def derive_symptomatic_side_outcome(
    dataframe: pd.DataFrame,
    output_column: str,
    right_column: str | None,
    left_column: str | None,
    higher_is_worse: bool = False,
) -> pd.DataFrame:
    """
    Derive a symptomatic side outcome variable by selecting the more
    symptomatic knee per patient based on the outcome value itself.

    For scales where lower = worse (e.g. KOOS, difficulty items), the side
    with the lower value is selected. For scales where higher = worse
    (e.g. pain severity, pain in bed), the side with the higher value is
    selected. Missing values on one side cause the other side to be
    selected automatically.

    Variables that exist on only one side (right or left column is None)
    are passed through directly without selection logic.

    :param dataframe: Input DataFrame containing one row per patient.
    :param output_column: Name for the derived symptomatic side column.
    :param right_column: Column name for the right knee value, or None
        if the variable only exists for one side.
    :param left_column: Column name for the left knee value, or None
        if the variable only exists for one side.
    :param higher_is_worse: If True, the higher value is selected
        (pain severity scales). If False, the lower value is selected
        (difficulty/function scales).
    :returns: DataFrame with the derived symptomatic side column appended.
    """
    working_dataframe = dataframe.copy()

    # Single-side variables — pass through directly
    if right_column is None and left_column is not None:
        working_dataframe[output_column] = working_dataframe[left_column]
        return working_dataframe
    if left_column is None and right_column is not None:
        working_dataframe[output_column] = working_dataframe[right_column]
        return working_dataframe

    value_right = working_dataframe[right_column]
    value_left = working_dataframe[left_column]

    if higher_is_worse:
        # Higher value = more symptomatic — select the higher side
        use_right = value_right.fillna(-np.inf) >= value_left.fillna(-np.inf)
    else:
        # Lower value = more symptomatic — select the lower side
        use_right = value_right.fillna(np.inf) <= value_left.fillna(np.inf)

    working_dataframe[output_column] = np.where(
        use_right, value_right, value_left
    )

    return working_dataframe


def fit_explanatory_lasso_model(
    training_predictors: pd.DataFrame,
    training_outcome: pd.Series,
    predictor_columns: list[str],
    covariate_columns: list[str] = COVARIATE_COLUMNS,
) -> dict:
    """
    Fit the two-block explanatory model using LASSO regularisation on the
    activity predictor block.

    The baseline model (covariates only, plain OLS) and the full model
    (covariates + LASSO-regularised activity predictors) are both fitted.
    The incremental R² (full minus baseline, evaluated via cross-validation)
    is the key explanatory result.

    Covariates are never regularised — they are passed through as-is.
    Activity predictors are standardised before fitting.

    :param training_predictors: Training set DataFrame containing both
        covariate and activity predictor columns.
    :param training_outcome: Training set outcome Series.
    :param predictor_columns: List of activity predictor column names
        (these will be standardised and regularised).
    :param covariate_columns: List of covariate column names (these are
        included without regularisation).
    :returns: Dictionary with keys:
        - ``baseline_r2_cv``: Cross-validated R² of the baseline model.
        - ``full_r2_cv``: Cross-validated R² of the full LASSO model.
        - ``incremental_r2``: Difference (full minus baseline).
        - ``selected_predictors``: List of activity predictors with
          non-zero LASSO coefficients.
        - ``lasso_coefficients``: Series of non-zero coefficients mapped
          to predictor names.
        - ``optimal_alpha``: The regularisation strength selected by CV.
    """

    covariate_training = training_predictors[covariate_columns]
    activity_training = training_predictors[predictor_columns]

    # --- Baseline model: covariates only, plain OLS ---
    baseline_model = LinearRegression()
    baseline_scores = cross_val_score(
        estimator=baseline_model,
        X=covariate_training,
        y=training_outcome,
        cv=KFold(
            n_splits=CROSS_VALIDATION_FOLDS,
            shuffle=True,
            random_state=RANDOM_STATE,
        ),
        scoring="r2",
    )
    baseline_r2_cv = baseline_scores.mean()

    # --- Full model: covariates (unscaled) + standardised activity predictors ---
    scaler = StandardScaler()
    activity_training_scaled = scaler.fit_transform(activity_training)

    # Concatenate covariates with scaled activity predictors
    full_training_matrix = np.hstack(
        [covariate_training.values, activity_training_scaled]
    )

    lasso_model = LassoCV(
        cv=CROSS_VALIDATION_FOLDS,
        random_state=RANDOM_STATE,
        max_iter=50_000,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        lasso_model.fit(full_training_matrix, training_outcome)

    full_scores = cross_val_score(
        estimator=lasso_model,
        X=full_training_matrix,
        y=training_outcome,
        cv=KFold(
            n_splits=CROSS_VALIDATION_FOLDS,
            shuffle=True,
            random_state=RANDOM_STATE,
        ),
        scoring="r2",
    )
    full_r2_cv = full_scores.mean()

    # Extract non-zero activity predictor coefficients (skip covariate block)
    number_of_covariates = len(covariate_columns)
    activity_coefficients = lasso_model.coef_[number_of_covariates:]
    non_zero_mask = activity_coefficients != 0

    selected_predictors = [
        column
        for column, is_selected in zip(predictor_columns, non_zero_mask)
        if is_selected
    ]
    lasso_coefficients = pd.Series(
        data=activity_coefficients[non_zero_mask],
        index=selected_predictors,
        name="lasso_coefficient",
    ).sort_values(key=abs, ascending=False)

    incremental_r2 = full_r2_cv - baseline_r2_cv

    print(
        f"\n--- Stage 1: Explanatory LASSO model ---\n"
        f"  Baseline R² (covariates only, 10-fold CV) : {baseline_r2_cv:.4f}\n"
        f"  Full R² (covariates + activity, 10-fold CV): {full_r2_cv:.4f}\n"
        f"  Incremental R² (activity block)           : {incremental_r2:.4f}\n"
        f"  Optimal LASSO alpha                       : {lasso_model.alpha_:.6f}\n"
        f"  Activity predictors selected (non-zero)   : "
        f"{len(selected_predictors)} / {len(predictor_columns)}"
    )

    if selected_predictors:
        print("\n  Non-zero LASSO coefficients (ranked by absolute magnitude):")
        for predictor_name, coefficient_value in lasso_coefficients.items():
            print(f"    {predictor_name:<45} {coefficient_value:+.4f}")

    return {
        "baseline_r2_cv": baseline_r2_cv,
        "full_r2_cv": full_r2_cv,
        "incremental_r2": incremental_r2,
        "selected_predictors": selected_predictors,
        "lasso_coefficients": lasso_coefficients,
        "optimal_alpha": lasso_model.alpha_,
    }


def fit_predictive_elastic_net_model(
    training_predictors: pd.DataFrame,
    training_outcome: pd.Series,
    test_predictors: pd.DataFrame,
    test_outcome: pd.Series,
    predictor_columns: list[str],
    covariate_columns: list[str] = COVARIATE_COLUMNS,
) -> dict:
    """
    Fit an Elastic Net model for maximum predictive accuracy and evaluate
    it on the held-out test set.

    Covariates are included unregularised. Activity predictors are
    standardised. Alpha and l1_ratio are jointly tuned via cross-validation
    on the training set only.

    :param training_predictors: Training set predictor DataFrame.
    :param training_outcome: Training set outcome Series.
    :param test_predictors: Held-out test set predictor DataFrame.
    :param test_outcome: Held-out test set outcome Series.
    :param predictor_columns: Activity predictor column names.
    :param covariate_columns: Covariate column names.
    :returns: Dictionary with keys ``rmse``, ``mae``, ``r2`` evaluated
        on the held-out test set, plus ``optimal_alpha`` and
        ``optimal_l1_ratio``.
    """
    covariate_training = training_predictors[covariate_columns].values
    activity_training = training_predictors[predictor_columns].values
    covariate_test = test_predictors[covariate_columns].values
    activity_test = test_predictors[predictor_columns].values

    scaler = StandardScaler()
    activity_training_scaled = scaler.fit_transform(activity_training)
    activity_test_scaled = scaler.transform(activity_test)

    full_training_matrix = np.hstack([covariate_training, activity_training_scaled])
    full_test_matrix = np.hstack([covariate_test, activity_test_scaled])

    elastic_net_model = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
        cv=CROSS_VALIDATION_FOLDS,
        random_state=RANDOM_STATE,
        max_iter=50_000,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        elastic_net_model.fit(full_training_matrix, training_outcome)

    test_predictions = elastic_net_model.predict(full_test_matrix)

    rmse = np.sqrt(mean_squared_error(test_outcome, test_predictions))
    mae = mean_absolute_error(test_outcome, test_predictions)
    r2 = r2_score(test_outcome, test_predictions)

    print(
        f"\n--- Stage 2a: Elastic Net (held-out test set) ---\n"
        f"  R²   : {r2:.4f}\n"
        f"  RMSE : {rmse:.4f}\n"
        f"  MAE  : {mae:.4f}\n"
        f"  Optimal alpha    : {elastic_net_model.alpha_:.6f}\n"
        f"  Optimal l1 ratio : {elastic_net_model.l1_ratio_:.2f}"
    )

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "optimal_alpha": elastic_net_model.alpha_,
        "optimal_l1_ratio": elastic_net_model.l1_ratio_,
    }


def fit_predictive_random_forest_model(
    training_predictors: pd.DataFrame,
    training_outcome: pd.Series,
    test_predictors: pd.DataFrame,
    test_outcome: pd.Series,
    predictor_columns: list[str],
    covariate_columns: list[str] = COVARIATE_COLUMNS,
) -> dict:
    """
    Fit a Random Forest regressor for maximum predictive accuracy and
    evaluate it on the held-out test set.

    Hyperparameters (number of trees, max depth, min samples per leaf)
    are fixed at sensible defaults for a dataset of this size. Feature
    importances (mean decrease in impurity) are reported for the activity
    predictor block.

    :param training_predictors: Training set predictor DataFrame.
    :param training_outcome: Training set outcome Series.
    :param test_predictors: Held-out test set predictor DataFrame.
    :param test_outcome: Held-out test set outcome Series.
    :param predictor_columns: Activity predictor column names.
    :param covariate_columns: Covariate column names.
    :returns: Dictionary with keys ``rmse``, ``mae``, ``r2`` on the
        held-out test set, plus ``feature_importances`` as a Series
        sorted by importance descending.
    """
    all_predictor_columns = covariate_columns + predictor_columns

    training_matrix = training_predictors[all_predictor_columns].values
    test_matrix = test_predictors[all_predictor_columns].values

    random_forest_model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    random_forest_model.fit(training_matrix, training_outcome)

    test_predictions = random_forest_model.predict(test_matrix)

    rmse = np.sqrt(mean_squared_error(test_outcome, test_predictions))
    mae = mean_absolute_error(test_outcome, test_predictions)
    r2 = r2_score(test_outcome, test_predictions)

    feature_importances = pd.Series(
        data=random_forest_model.feature_importances_,
        index=all_predictor_columns,
        name="importance",
    ).sort_values(ascending=False)

    print(
        f"\n--- Stage 2b: Random Forest (held-out test set) ---\n"
        f"  R²   : {r2:.4f}\n"
        f"  RMSE : {rmse:.4f}\n"
        f"  MAE  : {mae:.4f}\n"
        f"\n  Top 10 feature importances:"
    )
    for feature_name, importance_value in feature_importances.head(10).items():
        print(f"    {feature_name:<45} {importance_value:.4f}")

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "feature_importances": feature_importances,
    }


def summarise_pipeline_results(
    outcome_column: str,
    explanatory_results: dict,
    elastic_net_results: dict,
    random_forest_results: dict,
) -> pd.DataFrame:
    """
    Compile a summary table of results across both stages and all models.

    :param outcome_column: Name of the outcome variable modelled.
    :param explanatory_results: Output of ``fit_explanatory_lasso_model``.
    :param elastic_net_results: Output of ``fit_predictive_elastic_net_model``.
    :param random_forest_results: Output of ``fit_predictive_random_forest_model``.
    :returns: Single-row DataFrame with one column per metric, suitable
        for concatenation across multiple outcome variables.
    """
    summary = {
        "outcome_variable": outcome_column,
        "stage_1_baseline_r2_cv": explanatory_results["baseline_r2_cv"],
        "stage_1_full_r2_cv": explanatory_results["full_r2_cv"],
        "stage_1_incremental_r2": explanatory_results["incremental_r2"],
        "stage_1_selected_predictor_count": len(
            explanatory_results["selected_predictors"]
        ),
        "stage_2_elastic_net_r2_test": elastic_net_results["r2"],
        "stage_2_elastic_net_rmse_test": elastic_net_results["rmse"],
        "stage_2_elastic_net_mae_test": elastic_net_results["mae"],
        "stage_2_random_forest_r2_test": random_forest_results["r2"],
        "stage_2_random_forest_rmse_test": random_forest_results["rmse"],
        "stage_2_random_forest_mae_test": random_forest_results["mae"],
    }

    return pd.DataFrame([summary])


def run_pipeline_for_outcome_group(
    group_name: str,
    merged_dataframe: pd.DataFrame,
    merged_dataframe_with_symptomatic_side: pd.DataFrame,
    activity_predictor_columns: list[str],
    output_path,
) -> pd.DataFrame:
    """
    Run the two-stage predictive modelling pipeline for a named outcome group.

    Available groups:
        - ``pain``          : KOOS pain, ICOAP, pain severity, pain in bed,
                              global rating
        - ``function``      : WOMAC difficulty items, KOOS function items,
                              20m walk, chair stand, 400m walk
        - ``adl``           : LLDI limitation total, ADL items
        - ``activity``      : PASE household and leisure items
        - ``catastrophizing``: Coping strategies catastrophizing score
        - ``all``           : All outcome groups combined

    :param group_name: Name of the outcome group to model. Must be one of
        the groups listed above.
    :param merged_dataframe: Full merged DataFrame after whitespace cleaning
        and ``kl_grade_index_knee`` derivation. Used for bilateral outcomes.
    :param merged_dataframe_with_symptomatic_side: DataFrame with all
        symptomatic side outcome columns derived. Used for knee-specific
        outcomes.
    :param activity_predictor_columns: List of activity predictor column
        names to use in the modelling pipeline.
    :param output_path: Path object pointing to the output directory where
        the results CSV will be saved.
    :returns: DataFrame with one row per outcome variable containing all
        Stage 1 and Stage 2 metrics.
    :raises ValueError: If ``group_name`` is not a recognised group.
    """
    outcome_groups = {
        "pain": {
            "symptomatic_side": [
                "koos_pain_symptomatic_side",
                "icoap_total_symptomatic_side",
                "pain_7day_symptomatic_side",
                "pain_bed_symptomatic_side",
                "global_rating_symptomatic_side",
            ],
            "bilateral": [],
        },
        "self_reported_function": {
            "symptomatic_side": [
                "dirkn1_symptomatic_side",
                "dirkn2_symptomatic_side",
                "dirkn3_symptomatic_side",
                "dirkn4_symptomatic_side",
                "dirkn5_symptomatic_side",
                "dirkn6_symptomatic_side",
                "dirkn7_symptomatic_side",
                "dirkn8_symptomatic_side",
                "dirkn9_symptomatic_side",
                "dirkn13_symptomatic_side",
                "dirkn15_symptomatic_side",
                "dirkn16_symptomatic_side",
                "dirkn17_symptomatic_side",
                "dilkn10_symptomatic_side",
                "dilkn12_symptomatic_side",
            ],
            "bilateral": [
                "V06KOOSFX1",   # Either knee: squatting
                "V06KOOSFX2",   # Either knee: running
                "V06KOOSFX4",   # Either knee: twisting/pivoting
                "V06KOOSFX5",   # Either knee: kneeling
            ],
        },
        "function": {
             "symptomatic_side": [],
            "bilateral": [
                "V0620MPACE",   # 20m walk pace
                "V06CSPACE",    # Chair stand pace
                "V06STEPST1",   # 20m walk steps
                "V06400MTIM",   # 400m walk time
                "V06400MTR",    # 400m walk meters
                "V06CSTSGL",    # Single chair stand
            ],
        },
        "adl": {
            "symptomatic_side": [],
            "bilateral": [
                "V06LLD6B",     # LLDI: limited in active recreation
                "V06LLD6A",     # LLDI: frequency active recreation
                "V06LLDILST",   # LLDI: limitation total score
                "V06LLD10B",    # LLDI: limited in fitness program
                "V06ADL7A",     # ADL: equipment use getting in/out of bed
            ],
        },
        "activity": {
            "symptomatic_side": [],
            "bilateral": [
                "V06HOUACT6",   # Household activities: caring for another
                "V06PASE6HR",   # Leisure: muscle strength hours/day
            ],
        },
        "catastrophizing": {
            "symptomatic_side": [],
            "bilateral": [
                "V06CSQCAT",    # Coping strategies: catastrophizing score
            ],
        },
    }

    outcome_groups["all"] = {
        "symptomatic_side": [
            outcome
            for group in outcome_groups.values()
            for outcome in group["symptomatic_side"]
        ],
        "bilateral": [
            outcome
            for group in outcome_groups.values()
            for outcome in group["bilateral"]
        ],
    }

    if group_name not in outcome_groups:
        raise ValueError(
            f"Unknown group '{group_name}'. "
            f"Available groups: {sorted(outcome_groups.keys())}"
        )

    selected_group = outcome_groups[group_name]
    all_outcome_columns = (
        selected_group["symptomatic_side"]
        + selected_group["bilateral"]
    )

    print(
        f"\nRunning pipeline for group '{group_name}' "
        f"({len(all_outcome_columns)} outcome variables)"
    )

    all_results = []

    for outcome_column in all_outcome_columns:

        print(f"\n{'='*60}")
        print(f"Modelling outcome: {outcome_column}")
        print(f"{'='*60}")

        source_dataframe = (
            merged_dataframe_with_symptomatic_side
            if outcome_column in selected_group["symptomatic_side"]
            else merged_dataframe
        )

        try:
            (
                training_predictors,
                training_outcome,
                test_predictors,
                test_outcome,
            ) = prepare_modelling_dataset(
                dataframe=source_dataframe,
                outcome_column=outcome_column,
                predictor_columns=activity_predictor_columns,
            )

            explanatory_results = fit_explanatory_lasso_model(
                training_predictors=training_predictors,
                training_outcome=training_outcome,
                predictor_columns=activity_predictor_columns,
            )

            elastic_net_results = fit_predictive_elastic_net_model(
                training_predictors=training_predictors,
                training_outcome=training_outcome,
                test_predictors=test_predictors,
                test_outcome=test_outcome,
                predictor_columns=activity_predictor_columns,
            )

            random_forest_results = fit_predictive_random_forest_model(
                training_predictors=training_predictors,
                training_outcome=training_outcome,
                test_predictors=test_predictors,
                test_outcome=test_outcome,
                predictor_columns=activity_predictor_columns,
            )

            results_summary = summarise_pipeline_results(
                outcome_column=outcome_column,
                explanatory_results=explanatory_results,
                elastic_net_results=elastic_net_results,
                random_forest_results=random_forest_results,
            )

            all_results.append(results_summary)

        except ValueError as error:
            print(f"  Skipped '{outcome_column}': {error}")

    combined_results = pd.concat(all_results, ignore_index=True)

    output_filename = f"modelling_results_{group_name}.csv"
    combined_results.to_csv(
        output_path / output_filename,
        index=False,
    )
    print(
        f"\nGroup '{group_name}' complete — "
        f"{len(combined_results)} outcomes modelled. "
        f"Results saved to: {output_filename}"
    )

    return combined_results

In [13]:
MERGED_DATAFRAME_SEPARATOR = "|"

merged_dataframe = pd.read_csv(
    output_path / "summary_metrics_06.csv",
    sep=MERGED_DATAFRAME_SEPARATOR,
    na_values=[" ", ""],
)

# Replace any remaining whitespace-only strings with NaN across all columns
merged_dataframe = merged_dataframe.apply(
    lambda column: column.map(
        lambda value: np.nan
        if isinstance(value, str) and value.strip() == ""
        else value
    )
)

print(f"Loaded merged DataFrame: {merged_dataframe.shape}")

# Derive kl_grade_index_knee for the full dataframe — needed for
# train/test stratification across all outcomes, including bilateral ones
merged_dataframe["kl_grade_index_knee"] = np.where(
    merged_dataframe[KL_GRADE_RIGHT_COLUMN].fillna(-1)
    >= merged_dataframe[KL_GRADE_LEFT_COLUMN].fillna(-1),
    merged_dataframe[KL_GRADE_RIGHT_COLUMN],
    merged_dataframe[KL_GRADE_LEFT_COLUMN],
)

PREDICTOR_COLUMNS = [
    "V06AACNT",
    "V06AALTMNT",
    "V06AAMDMNT",
    "V06AAMVMNT",
    "V06AAVMNT",
    "V06AAMVBMT",
    "V06AAVBMT",
    "V06ANVDAYS",
    "V06AACSM03",
    "V06ADHHS8",
    "V06ADHHSD8",
    "wake_minute_mean",
    "wake_minute_sd",
    "V06CESD11",
    "sleep_minute_mean",
    "sleep_minute_sd",
    "wear_duration_mean",
    "valid_days",
    "weekly_guideline_gap_minutes",
    "mean_sedentary_bout_count",
    "mean_sedentary_bout_mean_duration",
    "mean_sedentary_bout_max_duration",
    "mean_sedentary_bout_total_minutes",
    "mean_light_bout_count",
    "mean_light_bout_mean_duration",
    "mean_light_bout_max_duration",
    "mean_light_bout_total_minutes",
    "mean_moderate_bout_count",
    "mean_moderate_bout_mean_duration",
    "mean_moderate_bout_max_duration",
    "mean_moderate_bout_total_minutes",
    "mean_vigorous_bout_count",
    "mean_vigorous_bout_mean_duration",
    "mean_vigorous_bout_max_duration",
    "mean_vigorous_bout_total_minutes",
    "mean_mvpa_bout_count",
    "mean_mvpa_bout_mean_duration",
    "mean_mvpa_bout_max_duration",
    "mean_mvpa_bout_total_minutes",
    "mean_active_bout_count",
    "mean_active_bout_mean_duration",
    "mean_active_bout_max_duration",
    "mean_active_bout_total_minutes",
]

ACTIVITY_PREDICTOR_COLUMNS = [
    column
    for column in PREDICTOR_COLUMNS
    if column not in COVARIATE_COLUMNS
]

# ---------------------------------------------------------------------------
# Bilateral outcome variable pairs
# ---------------------------------------------------------------------------

# Pairs where LOWER value = worse (KOOS only — 0–100 higher=better)
BILATERAL_PAIRS_LOWER_IS_WORSE = {
    "koos_pain_symptomatic_side":  ("V06KOOSKPR",  "V06KOOSKPL"),
    "dilkn10_symptomatic_side":    (None,           "V06DILKN10"),  # left only
    "dilkn12_symptomatic_side":    (None,           "V06DILKN12"),  # left only
}

# Pairs where HIGHER value = worse (pain severity, ICOAP, WOMAC difficulty items)
BILATERAL_PAIRS_HIGHER_IS_WORSE = {
    "icoap_total_symptomatic_side":  ("V06ICPTSKR",  "V06ICPTSKL"),
    "pain_7day_symptomatic_side":    ("V06P7RKRCV",  "V06P7LKRCV"),
    "pain_bed_symptomatic_side":     ("V06WPRKN3",   "V06WPLKN3"),
    "global_rating_symptomatic_side": ("V06KGLRS",   None),
    "dirkn1_symptomatic_side":       ("V06DIRKN1",   "V06DILKN1"),
    "dirkn2_symptomatic_side":       ("V06DIRKN2",   "V06DILKN2"),
    "dirkn3_symptomatic_side":       ("V06DIRKN3",   "V06DILKN3"),
    "dirkn4_symptomatic_side":       ("V06DIRKN4",   "V06DILKN4"),
    "dirkn5_symptomatic_side":       ("V06DIRKN5",   "V06DILKN5"),
    "dirkn6_symptomatic_side":       ("V06DIRKN6",   "V06DILKN6"),
    "dirkn7_symptomatic_side":       ("V06DIRKN7",   "V06DILKN7"),
    "dirkn8_symptomatic_side":       ("V06DIRKN8",   "V06DILKN8"),
    "dirkn9_symptomatic_side":       ("V06DIRKN9",   "V06DILKN9"),
    "dirkn13_symptomatic_side":      ("V06DIRKN13",  "V06DILKN13"),
    "dirkn15_symptomatic_side":      ("V06DIRKN15",  "V06DILKN15"),
    "dirkn16_symptomatic_side":      ("V06DIRKN16",  "V06DILKN16"),
    "dirkn17_symptomatic_side":      ("V06DIRKN17",  "V06DILKN17"),
}

merged_dataframe_with_symptomatic_side = merged_dataframe.copy()

for output_column, (right_column, left_column) in BILATERAL_PAIRS_LOWER_IS_WORSE.items():
    merged_dataframe_with_symptomatic_side = derive_symptomatic_side_outcome(
        dataframe=merged_dataframe_with_symptomatic_side,
        output_column=output_column,
        right_column=right_column,
        left_column=left_column,
        higher_is_worse=False,
    )

for output_column, (right_column, left_column) in BILATERAL_PAIRS_HIGHER_IS_WORSE.items():
    merged_dataframe_with_symptomatic_side = derive_symptomatic_side_outcome(
        dataframe=merged_dataframe_with_symptomatic_side,
        output_column=output_column,
        right_column=right_column,
        left_column=left_column,
        higher_is_worse=True,
    )

print(
    f"Symptomatic side outcomes derived: "
    f"{len(BILATERAL_PAIRS_LOWER_IS_WORSE) + len(BILATERAL_PAIRS_HIGHER_IS_WORSE)}"
)

Loaded merged DataFrame: (1845, 130)
Symptomatic side outcomes derived: 20


### 1.2.1. Results for pain outcomes

In [14]:
pain_results = run_pipeline_for_outcome_group(
    group_name="pain",
    merged_dataframe=merged_dataframe,
    merged_dataframe_with_symptomatic_side=merged_dataframe_with_symptomatic_side,
    activity_predictor_columns=ACTIVITY_PREDICTOR_COLUMNS,
    output_path=output_path,
)


Running pipeline for group 'pain' (5 outcome variables)

Modelling outcome: koos_pain_symptomatic_side
Complete cases for 'koos_pain_symptomatic_side':
  Total patients  : 1845
  Complete cases  : 1810 (98.1 %)
  Dropped (missing): 35
Train/test split (stratified by KL grade):
  Training set : 1448 patients
  Test set     : 362 patients

--- Stage 1: Explanatory LASSO model ---
  Baseline R² (covariates only, 10-fold CV) : 0.1024
  Full R² (covariates + activity, 10-fold CV): 0.1268
  Incremental R² (activity block)           : 0.0243
  Optimal LASSO alpha                       : 0.035135
  Activity predictors selected (non-zero)   : 27 / 43

  Non-zero LASSO coefficients (ranked by absolute magnitude):
    mean_light_bout_total_minutes                 +5.6685
    mean_mvpa_bout_count                          +5.0583
    V06AALTMNT                                    -3.6780
    mean_moderate_bout_total_minutes              -3.3476
    V06CESD11                                     -2.7

### 1.2.2. Results for function outcomes

In [15]:
function_results = run_pipeline_for_outcome_group(
    group_name="function",
    merged_dataframe=merged_dataframe,
    merged_dataframe_with_symptomatic_side=merged_dataframe_with_symptomatic_side, activity_predictor_columns=ACTIVITY_PREDICTOR_COLUMNS,
    output_path=output_path,
)



Running pipeline for group 'function' (6 outcome variables)

Modelling outcome: V0620MPACE
Complete cases for 'V0620MPACE':
  Total patients  : 1845
  Complete cases  : 1767 (95.8 %)
  Dropped (missing): 78
Train/test split (stratified by KL grade):
  Training set : 1413 patients
  Test set     : 354 patients

--- Stage 1: Explanatory LASSO model ---
  Baseline R² (covariates only, 10-fold CV) : 0.1910
  Full R² (covariates + activity, 10-fold CV): 0.2520
  Incremental R² (activity block)           : 0.0609
  Optimal LASSO alpha                       : 0.005160
  Activity predictors selected (non-zero)   : 8 / 43

  Non-zero LASSO coefficients (ranked by absolute magnitude):
    mean_moderate_bout_count                      +0.0288
    mean_mvpa_bout_max_duration                   +0.0184
    mean_moderate_bout_max_duration               +0.0100
    V06ANVDAYS                                    +0.0098
    V06CESD11                                     -0.0054
    mean_vigorous_bout_co

### 1.2.3. Results for ADL outcomes

In [16]:
adl_results = run_pipeline_for_outcome_group(
    group_name="adl",
    merged_dataframe=merged_dataframe,
    merged_dataframe_with_symptomatic_side=merged_dataframe_with_symptomatic_side,
    activity_predictor_columns=ACTIVITY_PREDICTOR_COLUMNS,
    output_path=output_path,
)


Running pipeline for group 'adl' (5 outcome variables)

Modelling outcome: V06LLD6B
Complete cases for 'V06LLD6B':
  Total patients  : 1845
  Complete cases  : 1801 (97.6 %)
  Dropped (missing): 44
Train/test split (stratified by KL grade):
  Training set : 1440 patients
  Test set     : 361 patients

--- Stage 1: Explanatory LASSO model ---
  Baseline R² (covariates only, 10-fold CV) : 0.0396
  Full R² (covariates + activity, 10-fold CV): 0.1355
  Incremental R² (activity block)           : 0.0959
  Optimal LASSO alpha                       : 0.003032
  Activity predictors selected (non-zero)   : 23 / 43

  Non-zero LASSO coefficients (ranked by absolute magnitude):
    V06CESD11                                     -0.2892
    V06AACNT                                      +0.2691
    mean_active_bout_total_minutes                -0.2042
    V06AAMVBMT                                    -0.1156
    V06AAVBMT                                     -0.0934
    mean_mvpa_bout_max_duration  

### 1.2.4. Results for activity outcomes

In [17]:
activity_results = run_pipeline_for_outcome_group(
    group_name="activity",
    merged_dataframe=merged_dataframe,
    merged_dataframe_with_symptomatic_side=merged_dataframe_with_symptomatic_side,activity_predictor_columns=ACTIVITY_PREDICTOR_COLUMNS,
    output_path=output_path,
)


Running pipeline for group 'activity' (2 outcome variables)

Modelling outcome: V06HOUACT6
Complete cases for 'V06HOUACT6':
  Total patients  : 1845
  Complete cases  : 1768 (95.8 %)
  Dropped (missing): 77
Train/test split (stratified by KL grade):
  Training set : 1414 patients
  Test set     : 354 patients

--- Stage 1: Explanatory LASSO model ---
  Baseline R² (covariates only, 10-fold CV) : -0.0067
  Full R² (covariates + activity, 10-fold CV): 0.0080
  Incremental R² (activity block)           : 0.0146
  Optimal LASSO alpha                       : 0.017180
  Activity predictors selected (non-zero)   : 6 / 43

  Non-zero LASSO coefficients (ranked by absolute magnitude):
    V06AALTMNT                                    +0.0466
    mean_moderate_bout_total_minutes              -0.0194
    mean_mvpa_bout_total_minutes                  -0.0053
    mean_light_bout_mean_duration                 +0.0036
    wake_minute_sd                                +0.0033
    mean_mvpa_bout_max_d

### 1.2.5. Results for catastrophizing outcomes

In [18]:
catastrophizing_results = run_pipeline_for_outcome_group(
    group_name="catastrophizing",
    merged_dataframe=merged_dataframe,
    merged_dataframe_with_symptomatic_side=merged_dataframe_with_symptomatic_side,
    activity_predictor_columns=ACTIVITY_PREDICTOR_COLUMNS,
    output_path=output_path,
)


Running pipeline for group 'catastrophizing' (1 outcome variables)

Modelling outcome: V06CSQCAT
Complete cases for 'V06CSQCAT':
  Total patients  : 1845
  Complete cases  : 1749 (94.8 %)
  Dropped (missing): 96
Train/test split (stratified by KL grade):
  Training set : 1399 patients
  Test set     : 350 patients

--- Stage 1: Explanatory LASSO model ---
  Baseline R² (covariates only, 10-fold CV) : 0.0126
  Full R² (covariates + activity, 10-fold CV): 0.0443
  Incremental R² (activity block)           : 0.0317
  Optimal LASSO alpha                       : 0.003321
  Activity predictors selected (non-zero)   : 22 / 43

  Non-zero LASSO coefficients (ranked by absolute magnitude):
    mean_sedentary_bout_total_minutes             -0.2082
    mean_light_bout_count                         +0.1897
    V06CESD11                                     +0.1772
    V06AALTMNT                                    -0.1577
    sleep_minute_mean                             +0.1082
    V06ADHHS8      

### 1.2.6. Results for self-reported function outcomes

In [19]:
self_reported_function_results = run_pipeline_for_outcome_group(
    group_name="self_reported_function",
    merged_dataframe=merged_dataframe,
    merged_dataframe_with_symptomatic_side=merged_dataframe_with_symptomatic_side,
    activity_predictor_columns=ACTIVITY_PREDICTOR_COLUMNS,
    output_path=output_path,
)


Running pipeline for group 'self_reported_function' (19 outcome variables)

Modelling outcome: dirkn1_symptomatic_side
Complete cases for 'dirkn1_symptomatic_side':
  Total patients  : 1845
  Complete cases  : 1804 (97.8 %)
  Dropped (missing): 41
Train/test split (stratified by KL grade):
  Training set : 1443 patients
  Test set     : 361 patients

--- Stage 1: Explanatory LASSO model ---
  Baseline R² (covariates only, 10-fold CV) : 0.0959
  Full R² (covariates + activity, 10-fold CV): 0.1167
  Incremental R² (activity block)           : 0.0208
  Optimal LASSO alpha                       : 0.017740
  Activity predictors selected (non-zero)   : 9 / 43

  Non-zero LASSO coefficients (ranked by absolute magnitude):
    V06CESD11                                     +0.0761
    mean_mvpa_bout_count                          -0.0466
    wake_minute_sd                                +0.0463
    mean_moderate_bout_max_duration               -0.0327
    mean_moderate_bout_mean_duration      

### 1.2.7. Summary table across all outcome groups

In [20]:
def build_combined_summary_table(
        *,
        group_result_frames: dict[str, pd.DataFrame],
        sort_by_column: str = "stage_1_incremental_r2",
) -> pd.DataFrame:
    """
    Concatenate the per-group result frames into a single summary table,
    tagging each row with the outcome group it belongs to.

    Each input frame is the return value of
    ``run_pipeline_for_outcome_group`` and already contains one row per
    outcome variable with all Stage 1 and Stage 2 metrics. This function
    prepends an ``outcome_group`` column, stacks the frames, and sorts the
    result so the strongest activity signal appears first.

    :param group_result_frames: Mapping of outcome group name to the
        result frame returned for that group.
    :param sort_by_column: Column to sort the combined table by, in
        descending order. Defaults to the Stage 1 incremental R², which
        quantifies the unique variance explained by the activity block.
    :returns: Combined DataFrame with one row per outcome variable across
        all groups, sorted by ``sort_by_column`` descending.
    """
    labelled_frames = []

    for group_name, result_frame in group_result_frames.items():
        labelled_frame = result_frame.copy()
        labelled_frame.insert(loc=0, column="outcome_group", value=group_name)
        labelled_frames.append(labelled_frame)

    combined_table = pd.concat(labelled_frames, ignore_index=True)

    combined_table = combined_table.sort_values(
        by=sort_by_column,
        ascending=False,
    ).reset_index(drop=True)

    return combined_table


combined_summary_table = build_combined_summary_table(
    group_result_frames={
        "pain": pain_results,
        "function": function_results,
        "adl": adl_results,
        "activity": activity_results,
        "catastrophizing": catastrophizing_results,
        "self_reported_function": self_reported_function_results,
    },
    sort_by_column="stage_1_incremental_r2",
)

combined_summary_table.to_csv(
    output_path / "modelling_results_combined.csv",
    index=False,
)

print(
    f"Combined summary table: {len(combined_summary_table)} outcomes "
    f"across {combined_summary_table['outcome_group'].nunique()} groups"
)
print(combined_summary_table.to_string(index=False))

Combined summary table: 38 outcomes across 6 groups
         outcome_group               outcome_variable  stage_1_baseline_r2_cv  stage_1_full_r2_cv  stage_1_incremental_r2  stage_1_selected_predictor_count  stage_2_elastic_net_r2_test  stage_2_elastic_net_rmse_test  stage_2_elastic_net_mae_test  stage_2_random_forest_r2_test  stage_2_random_forest_rmse_test  stage_2_random_forest_mae_test
                   adl                     V06LLDILST                0.036322            0.148350                0.112028                                16                     0.144614                      14.131429                     11.992375                       0.121052                        14.324733                       12.204049
                   adl                       V06LLD6A                0.042488            0.138653                0.096165                                12                     0.181084                       1.068538                      0.857534                   

# 2. Predictive models with harmonic features and IV

## 2.1. Prepare modeling dataframe
Derives the columns required by the modeling pipeline from the summary_metrics_06 dataframe.

Derivations:
- kl_grade_index_knee: worse-knee KL grade, used as structural, covariate and train/test stratification key in every stage.
- employment_working: binary working indicator derived from V06CEMPLOY, used as a demographic covariate.
- Symptomatic-side outcome columns: for bilateral knee variables, the more symptomatic knee is selected based on the outcome value. Lower = worse for KOOS-type scales; higher = worse for pain severity and ICOAP scales.

In [21]:
# Constants

DEMOGRAPHIC_COVARIATE_COLUMNS = [
    "V06AGE",
    "V06BMI",
    "V06COMORB",
    "employment_working",
]

STRUCTURAL_COVARIATE_COLUMN = "kl_grade_index_knee"

FINAL_PREDICTOR_COLUMNS = [
    # Sleep / wear
    "wake_minute_mean",                # Mean wake time across valid days
    "wake_minute_sd",                  # Standard deviation of wake time
    "wear_duration_mean",              # Mean daily wear duration in minutes

    # Bout structure: sedentary
    "mean_sedentary_bout_count",
    "mean_sedentary_bout_mean_duration",
    "mean_sedentary_bout_max_duration",
    "mean_sedentary_bout_total_minutes",

    # Bout structure: light
    "mean_light_bout_max_duration",

    # Bout structure: MVPA
    "mean_mvpa_bout_count",
    "mean_mvpa_bout_mean_duration",

    # Bout structure: active
    "mean_active_bout_mean_duration",
    "mean_active_bout_max_duration",
    "mean_active_bout_total_minutes",

    # Harmonic features: overall
    "acrophase",                       # Overall acrophase retained — not involved in any redundant pair
    "interdaily_stability",            # IS retained as overall only — day-split not meaningful

    # Harmonic features: day-type specific
    "mesor_mean_curve_weekday",        # Mean activity level on weekdays
    "mesor_mean_curve_weekend",        # Mean activity level on weekends
    "amplitude_mean_curve_weekday",    # Peak-to-mean swing on weekdays
    "amplitude_mean_curve_weekend",    # Peak-to-mean swing on weekends
    "acrophase_mean_curve_weekday",    # Timing of activity peak on weekdays
    "acrophase_mean_curve_weekend",    # Timing of activity peak on weekends

    # IV: day-type specific
    "iv_weekday",                      # Within-day fragmentation on weekdays
    "iv_weekend",                      # Within-day fragmentation on weekends
    "iv_contrast",                     # iv_weekday - iv_weekend: work-related fragmentation burden
]

OUTCOME_GROUPS: dict[str, list[str]] = {
    "pain": [
        "koos_pain_symptomatic_side",      # V06KOOSKPR / V06KOOSKPL — lower is worse
        "icoap_total_symptomatic_side",    # V06ICPTSKR / V06ICPTSKL — higher is worse
        "pain_7day_symptomatic_side",      # V06P7RKRCV / V06P7LKRCV — higher is worse
        "pain_bed_symptomatic_side",       # V06WPRKN3  / V06WPLKN3  — higher is worse
        "global_rating_symptomatic_side",  # V06KGLRS only (right side) — higher is worse
    ],
    "function": [
        "V0620MPACE",    # 20m walk pace (m/s)
        "V06CSPACE",     # repeated chair stand pace (stands/sec)
        "V06STEPST1",    # 20m walk trial 1 number of steps
        "V06400MTIM",    # 400m walk total time at 400m or at stop
        # V06400MTR dropped: severely negative baseline R² in Stage 1 (-0.43),
        # Ridge test R² 0.037 — variable distribution appears problematic
        # V06CSTSGL dropped: LASSO incremental R² -0.008, inconsistent signal
    ],
    "adl": [
        "V06LLD6B",      # LLDI: extent feel limited in taking part in active recreation
        "V06LLD6A",      # LLDI: how often take part in active recreation
        "V06LLDILST",    # LLDI: limitation dimension total score (calculated)
        "V06LLD10B",     # LLDI: extent feel limited in taking part in regular fitness program
        # V06ADL7A dropped: LASSO incremental R² 0.002, zero predictors selected,
        # Ridge test R² -0.05 — no signal across both stages
    ],
    "catastrophizing": [
        "V06CSQCAT",     # CSQ: coping strategies questionnaire catastrophizing score
    ],
    "self_reported_function": [
        "dirkn1_symptomatic_side",    # down stairs
        "dirkn2_symptomatic_side",    # up stairs
        "dirkn3_symptomatic_side",    # stand from sitting
        "dirkn4_symptomatic_side",    # standing
        "dirkn5_symptomatic_side",    # bending
        "dirkn6_symptomatic_side",    # walking
        "dirkn7_symptomatic_side",    # in car / out of car
        "dirkn8_symptomatic_side",    # shopping
        "dirkn9_symptomatic_side",    # socks on
        "dirkn13_symptomatic_side",   # get in / out of bathtub
        "dirkn15_symptomatic_side",   # on / off toilet
        "dirkn16_symptomatic_side",   # heavy chores
        "dirkn17_symptomatic_side",   # light chores
        "dilkn10_symptomatic_side",   # get out of bed (left only)
        # dilkn12_symptomatic_side dropped: LASSO incremental R² -0.0008,
        # Ridge test R² 0.02 — no signal across both stages
        "V06KOOSFX1",                 # either knee: squatting
        "V06KOOSFX2",                 # either knee: running — keep, 44% complete cases
        "V06KOOSFX4",                 # either knee: twisting / pivoting
        "V06KOOSFX5",                 # either knee: kneeling
    ],
    "activity": [
        "V06HOUACT6",    # household activities: caring for another person, past 7 days
        # V06PASE6HR dropped: LASSO incremental R² -0.0007, Ridge test R² -0.028,
        # only 41% complete cases — no signal and insufficient data
    ],
}

# Execution

"""
Copy summary_metrics_06 into the modelling dataframe.
summary_metrics_06 has ID set as the index at this point in the notebook
from the harmonic regression merge. Reset it to a regular column so that
prepare_modelling_dataset can use it for joins and stratification.
"""

modelling_dataframe = summary_metrics_06.copy().reset_index()

print(
    f"Loaded from memory:  {modelling_dataframe.shape[0]:,} participants, "
    f"{modelling_dataframe.shape[1]} columns"
)

# Rename kl_grade to kl_grade_index_knee.
modelling_dataframe = modelling_dataframe.rename(
    columns={"kl_grade": STRUCTURAL_COVARIATE_COLUMN}
)

kl_missing = modelling_dataframe[STRUCTURAL_COVARIATE_COLUMN].isna().sum()
print(f"kl_grade_index_knee ready — missing in {kl_missing} participants")

# employment_group string labels "working" and "not working". A numeric binary column employment_working (1 = working, 0 = not working) is derived

modelling_dataframe["employment_working"] = modelling_dataframe["employment_group"].map(
    {"working": 1.0, "not working": 0.0}
)

employment_counts = modelling_dataframe["employment_working"].value_counts(dropna=False)
print(
    f"employment_working derived:\n"
    f"  Working     : {employment_counts.get(1.0, 0)}\n"
    f"  Not working : {employment_counts.get(0.0, 0)}\n"
    f"  Missing     : {modelling_dataframe['employment_working'].isna().sum()}"
)

# Derive symptomatic-side outcome columns. Derive_symptomatic_side_outcome is defined earlier in this notebook.
for output_column, (right_column, left_column) in BILATERAL_PAIRS_LOWER_IS_WORSE.items():
    modelling_dataframe = derive_symptomatic_side_outcome(
        dataframe=modelling_dataframe,
        output_column=output_column,
        right_column=right_column,
        left_column=left_column,
        higher_is_worse=False,
    )

for output_column, (right_column, left_column) in BILATERAL_PAIRS_HIGHER_IS_WORSE.items():
    modelling_dataframe = derive_symptomatic_side_outcome(
        dataframe=modelling_dataframe,
        output_column=output_column,
        right_column=right_column,
        left_column=left_column,
        higher_is_worse=True,
    )

print(
    f"Symptomatic-side outcomes derived: "
    f"{len(BILATERAL_PAIRS_LOWER_IS_WORSE) + len(BILATERAL_PAIRS_HIGHER_IS_WORSE)}"
)

# Verify that all predictor columns are present. Missing columns here will cause silent NaN propagation or KeyErrors in later stages, so fail loudly now.
missing_predictors = [
    column for column in FINAL_PREDICTOR_COLUMNS
    if column not in modelling_dataframe.columns
]

if missing_predictors:
    print(
        f"WARNING — the following predictor columns are missing from the "
        f"dataframe and will cause errors in later stages:\n"
        f"  {missing_predictors}"
    )
else:
    print(f"All {len(FINAL_PREDICTOR_COLUMNS)} predictor columns present.")

# Final shape and missingness summary.
print(
    f"\nModelling dataframe ready:\n"
    f"  Shape         : {modelling_dataframe.shape}\n"
    f"  Participants  : {len(modelling_dataframe):,}"
)

predictor_missingness = (
    modelling_dataframe[FINAL_PREDICTOR_COLUMNS]
    .isna()
    .sum()
    .sort_values(ascending=False)
)
predictors_with_missing = predictor_missingness[predictor_missingness > 0]

if predictors_with_missing.empty:
    print("  Predictor missingness: none")
else:
    print("  Predictor missingness (columns with any NaN):")
    for column_name, missing_count in predictors_with_missing.items():
        print(
            f"    {column_name:<45} {missing_count:>4} "
            f"({100 * missing_count / len(modelling_dataframe):.1f} %)"
        )

Loaded from memory:  1,845 participants, 132 columns
kl_grade_index_knee ready — missing in 1 participants
employment_working derived:
  Working     : 984
  Not working : 813
  Missing     : 48
Symptomatic-side outcomes derived: 20
All 24 predictor columns present.

Modelling dataframe ready:
  Shape         : (1845, 153)
  Participants  : 1,845
  Predictor missingness (columns with any NaN):
    iv_contrast                                     41 (2.2 %)
    iv_weekend                                      41 (2.2 %)
    acrophase_mean_curve_weekend                    41 (2.2 %)
    amplitude_mean_curve_weekend                    41 (2.2 %)
    mesor_mean_curve_weekend                        41 (2.2 %)


## 2.2. Models

### 2.2.1. LASSO

**Purpose**
LASSO (Least Absolute Shrinkage and Selection Operator) is used here in its explanatory role rather than as a pure predictive model. The key result is the incremental R² — how much additional variance in each outcome is explained by the activity predictor block on top of the demographic and structural covariates alone.

**Why LASSO for the explanatory stage**
LASSO applies L1 regularisation, which shrinks the coefficients of weak or redundant predictors exactly to zero. This produces a sparse solution that identifies which activity features carry independent signal after the covariate block is accounted for. The sparsity is methodologically useful for a thesis: it forces explicit feature selection rather than distributing signal across all predictors, and the selected predictors can be reported and interpreted directly.

**Two-block design**
The model is fitted in two blocks: Block 1 (baseline): demographic covariates only (age, BMI, comorbidity count, employment status), fitted with plain OLS. Block 2 (full): covariates (unregularised) + activity predictors (standardised and L1-regularised via LassoCV).
The incremental R² (full minus baseline, both evaluated via 10-fold cross-validation on the training set) is the primary reported metric. It quantifies the unique contribution of the accelerometry-derived activity features independent of clinical background characteristics.

KL grade is included as an additional structural covariate for all non-structural outcomes. It is excluded only when KL grade is itself the outcome (Stage 6).

**What is NOT reported here**
Held-out test set performance is not the focus of this stage. The predictive model families (Ridge, Elastic Net, Random Forest, XGBoost) in Stages 2–5 are evaluated on the held-out test set. LASSO is evaluated on the training set via cross-validation to preserve its explanatory interpretation.

**Output**
lasso_results: dict mapping outcome_group_name -> list of per-outcome result dicts. Each result dict contains:
   - outcome_column
   - baseline_r2_cv
   - full_r2_cv
   - incremental_r2
   - selected_predictors (list of non-zero activity predictors)
   - lasso_coefficients (Series ranked by absolute magnitude)
   - optimal_alpha

In [22]:
def prepare_modelling_dataset_for_stage(
    *,
    dataframe: pd.DataFrame,
    outcome_column: str,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    structural_covariate_column: str,
    include_structural_covariate: bool = True,
    test_set_fraction: float = 0.20,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    Prepare a clean train/test split for a single outcome variable.

    Rows with any missing value in the covariate, predictor, or outcome
    columns are dropped before splitting. The split is stratified on KL
    grade (binned into three groups: 0–1, 2–3, 4) to preserve the
    severity distribution across both sets.

    :param dataframe: Modelling dataframe with one row per participant.
    :param outcome_column: Name of the outcome variable to predict.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names included without regularisation.
    :param structural_covariate_column: KL grade column name.
    :param include_structural_covariate: Whether to include KL grade in
        the covariate block. Set to False only when KL grade is the
        outcome (Stage 6).
    :param test_set_fraction: Proportion of data held out for testing.
    :param cross_validation_folds: Number of folds for cross-validation.
    :param random_state: Random seed for reproducibility.
    :returns: Tuple of (training_predictors, training_outcome,
        test_predictors, test_outcome).
    :raises ValueError: If fewer than 100 complete cases remain after
        dropping missing values.
    """
    covariate_columns = list(demographic_covariate_columns)
    if include_structural_covariate:
        covariate_columns.append(structural_covariate_column)

    all_required_columns = covariate_columns + activity_predictor_columns + [outcome_column]
    complete_cases = dataframe[all_required_columns].dropna()

    complete_case_count = len(complete_cases)
    total_count = len(dataframe)

    print(
        f"  Complete cases : {complete_case_count:,} of {total_count:,} "
        f"({100 * complete_case_count / total_count:.1f} %)"
    )

    if complete_case_count < 100:
        raise ValueError(
            f"Only {complete_case_count} complete cases remain for "
            f"'{outcome_column}'. Modelling would be unreliable."
        )

    outcome_series = complete_cases[outcome_column]
    predictor_dataframe = complete_cases[covariate_columns + activity_predictor_columns]

    kl_grade_bins = pd.cut(
        complete_cases[structural_covariate_column],
        bins=[-0.5, 1.5, 3.5, 4.5],
        labels=["kl_0_1", "kl_2_3", "kl_4"],
    )

    (
        training_predictors,
        test_predictors,
        training_outcome,
        test_outcome,
    ) = train_test_split(
        predictor_dataframe,
        outcome_series,
        test_size=test_set_fraction,
        random_state=random_state,
        stratify=kl_grade_bins,
    )

    print(
        f"  Training set   : {len(training_predictors):,} participants\n"
        f"  Test set       : {len(test_predictors):,} participants"
    )

    return training_predictors, training_outcome, test_predictors, test_outcome


def run_lasso_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    structural_covariate_column: str,
    include_structural_covariate: bool = True,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit the two-block explanatory LASSO model for a single outcome and
    return all metrics in a result dictionary.

    Block 1 (baseline): demographic covariates only, plain OLS, evaluated
    via cross-validation to produce baseline_r2_cv.

    Block 2 (full): covariates (unregularised) concatenated with
    standardised activity predictors, regularised via LassoCV.
    Cross-validated R² on the training set produces full_r2_cv.

    The incremental R² (full_r2_cv minus baseline_r2_cv) is the primary
    reported metric: it quantifies the unique variance explained by the
    activity block after controlling for demographic and structural
    covariates.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names
        (standardised and regularised).
    :param demographic_covariate_columns: Demographic covariate column
        names (included unregularised in both blocks).
    :param structural_covariate_column: KL grade column name.
    :param include_structural_covariate: Whether to include KL grade as
        a covariate. Always True except for Stage 6.
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys baseline_r2_cv, full_r2_cv,
        incremental_r2, selected_predictors, lasso_coefficients,
        optimal_alpha, outcome_column, n_training, n_test.
    """
    covariate_columns = list(demographic_covariate_columns)
    if include_structural_covariate:
        covariate_columns.append(structural_covariate_column)

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        structural_covariate_column=structural_covariate_column,
        include_structural_covariate=include_structural_covariate,
    )

    covariate_training = training_predictors[covariate_columns]
    activity_training = training_predictors[activity_predictor_columns]

    # Block 1 — baseline: covariates only, plain OLS
    kfold = KFold(
        n_splits=cross_validation_folds,
        shuffle=True,
        random_state=random_state,
    )

    baseline_scores = cross_val_score(
        estimator=LinearRegression(),
        X=covariate_training,
        y=training_outcome,
        cv=kfold,
        scoring="r2",
    )
    baseline_r2_cv = float(baseline_scores.mean())

    # Block 2 — full: covariates (unscaled) + standardised activity predictors
    scaler = StandardScaler()
    activity_training_scaled = scaler.fit_transform(activity_training)

    full_training_matrix = np.hstack(
        [covariate_training.values, activity_training_scaled]
    )

    lasso_model = LassoCV(
        cv=cross_validation_folds,
        random_state=random_state,
        max_iter=50_000,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        lasso_model.fit(full_training_matrix, training_outcome)

    full_scores = cross_val_score(
        estimator=lasso_model,
        X=full_training_matrix,
        y=training_outcome,
        cv=KFold(
            n_splits=cross_validation_folds,
            shuffle=True,
            random_state=random_state,
        ),
        scoring="r2",
    )
    full_r2_cv = float(full_scores.mean())
    incremental_r2 = full_r2_cv - baseline_r2_cv

    # Extract non-zero activity predictor coefficients
    # The covariate block occupies the first len(covariate_columns) positions.
    # Activity coefficients start immediately after.
    number_of_covariates = len(covariate_columns)
    activity_coefficients = lasso_model.coef_[number_of_covariates:]
    non_zero_mask = activity_coefficients != 0

    selected_predictors = [
        column
        for column, is_selected in zip(activity_predictor_columns, non_zero_mask)
        if is_selected
    ]

    lasso_coefficients = pd.Series(
        data=activity_coefficients[non_zero_mask],
        index=selected_predictors,
        name="lasso_coefficient",
    ).sort_values(key=abs, ascending=False)

    print(
        f"\n  Baseline R² (covariates only, {cross_validation_folds}-fold CV) : "
        f"{baseline_r2_cv:.4f}\n"
        f"  Full R²     (covariates + activity)            : "
        f"{full_r2_cv:.4f}\n"
        f"  Incremental R² (activity block)                : "
        f"{incremental_r2:.4f}\n"
        f"  Optimal alpha                                  : "
        f"{lasso_model.alpha_:.6f}\n"
        f"  Predictors selected (non-zero)                 : "
        f"{len(selected_predictors)} / {len(activity_predictor_columns)}"
    )

    if selected_predictors:
        print("\n  Non-zero coefficients (ranked by absolute magnitude):")
        for predictor_name, coefficient_value in lasso_coefficients.items():
            print(f"    {predictor_name:<45} {coefficient_value:+.4f}")

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "baseline_r2_cv": baseline_r2_cv,
        "full_r2_cv": full_r2_cv,
        "incremental_r2": incremental_r2,
        "selected_predictors": selected_predictors,
        "lasso_coefficients": lasso_coefficients,
        "optimal_alpha": float(lasso_model.alpha_),
    }

#### Execute LASSO for all outcomes

In [23]:
RANDOM_STATE = 42
CROSS_VALIDATION_FOLDS = 10

lasso_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_lasso_for_outcome(
                outcome_column=outcome_column,
                dataframe=modelling_dataframe,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                structural_covariate_column=STRUCTURAL_COVARIATE_COLUMN,
                include_structural_covariate=True,
                cross_validation_folds=CROSS_VALIDATION_FOLDS,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    lasso_results[group_name] = group_results


Group: pain  (5 outcomes)

  Outcome: koos_pain_symptomatic_side
  --------------------------------------------------
  Complete cases : 1,736 of 1,845 (94.1 %)
  Training set   : 1,388 participants
  Test set       : 348 participants

  Baseline R² (covariates only, 10-fold CV) : 0.1201
  Full R²     (covariates + activity)            : 0.1354
  Incremental R² (activity block)                : 0.0153
  Optimal alpha                                  : 0.336356
  Predictors selected (non-zero)                 : 8 / 24

  Non-zero coefficients (ranked by absolute magnitude):
    mean_mvpa_bout_count                          +1.4730
    mean_sedentary_bout_total_minutes             +1.1071
    wake_minute_sd                                -0.8628
    iv_weekend                                    -0.7125
    interdaily_stability                          -0.5365
    mean_mvpa_bout_mean_duration                  +0.3174
    amplitude_mean_curve_weekday                  +0.1041
    iv_weekda

#### Lasso summary table

In [24]:
summary_rows = []

for group_name, group_results in lasso_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "baseline_r2_cv": round(result["baseline_r2_cv"], 4),
            "full_r2_cv": round(result["full_r2_cv"], 4),
            "incremental_r2": round(result["incremental_r2"], 4),
            "n_selected_predictors": len(result["selected_predictors"]),
            "optimal_alpha": round(result["optimal_alpha"], 6),
        })

lasso_summary_table = pd.DataFrame(summary_rows)

print("\nStage 1 — LASSO summary (sorted by incremental R²):")
print(
    lasso_summary_table
    .sort_values("incremental_r2", ascending=False)
    .to_string(index=False)
)


Stage 1 — LASSO summary (sorted by incremental R²):
                 group                        outcome  n_training  baseline_r2_cv  full_r2_cv  incremental_r2  n_selected_predictors  optimal_alpha
                   adl                       V06LLD6A        1386          0.0474      0.1440          0.0965                     20       0.004237
              function                     V06STEPST1        1364          0.1831      0.2747          0.0916                     14       0.031829
              function                     V0620MPACE        1364          0.1726      0.2258          0.0531                     12       0.002958
              function                     V06400MTIM        1248          0.1783      0.2285          0.0501                     12       0.698085
                   adl                     V06LLDILST        1352          0.0243      0.0600          0.0358                     15       0.148915
                   adl                       V06LLD6B      

### 2.2.2. Ridge regression

**Purpose**

Ridge regression is the first purely predictive model in the pipeline.Unlike Stage 1 LASSO, the goal here is not feature selection or incremental explanation — it is to assess how well the full predictor set predicts each outcome on unseen data, evaluated on the held-out test set.

**Why Ridge after LASSO**
 LASSO (L1) shrinks weak coefficients to exactly zero, producing a sparse solution. Ridge (L2) shrinks all coefficients continuously toward zero but retains every predictor in the model. This means Ridge can outperform LASSO when many predictors each carry a small amount of genuine signal — a situation that is plausible here given the moderate intercorrelations among bout-structure and harmonic features. Comparing Ridge test R² against LASSO full_r2_cv gives a first indication of whether sparsity or dense shrinkage better fits the data for each outcome.

 **Regularisation**
 The regularisation strength alpha is selected via RidgeCV using 10-fold cross-validation on the training set across a log-spaced grid from 0.001 to 1000. Activity predictors are standardised on the training set; the same scaler is applied to the test set. Covariates are passed through unscaled, consistent with the convention used in Stage 1.

 **Output**

 ridge_results: dict mapping outcome_group_name -> list of per-outcome result dicts. Each result dict contains:
   - outcome_column
   - n_training, n_test
   - cross_validated_r2 (training set, 10-fold CV)
   - test_r2
   - test_rmse
   - test_mae
   - optimal_alpha


In [25]:
def run_ridge_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    structural_covariate_column: str,
    include_structural_covariate: bool = True,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit a Ridge regression model with cross-validated alpha selection
    for a single outcome and return held-out test set metrics.

    Activity predictors are standardised on the training set; the same
    scaler is applied to the test set without refitting. Covariates are
    passed through unscaled. Alpha is selected from a log-spaced grid
    of 25 values between 0.001 and 1000 using RidgeCV with 10-fold CV
    on the training set.

    A second cross-validation pass using the selected alpha is run on
    the full training matrix to produce a cross_validated_r2 that is
    directly comparable to the baseline_r2_cv and full_r2_cv from
    Stage 1.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names
        (standardised before fitting).
    :param demographic_covariate_columns: Demographic covariate column
        names (passed through unscaled).
    :param structural_covariate_column: KL grade column name.
    :param include_structural_covariate: Whether to include KL grade as
        a covariate. Always True except for Stage 6.
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys outcome_column, n_training, n_test,
        cross_validated_r2, test_r2, test_rmse, test_mae, optimal_alpha.
    """
    covariate_columns = list(demographic_covariate_columns)
    if include_structural_covariate:
        covariate_columns.append(structural_covariate_column)

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        structural_covariate_column=structural_covariate_column,
        include_structural_covariate=include_structural_covariate,
    )

    scaler = StandardScaler()
    activity_training_scaled = scaler.fit_transform(
        training_predictors[activity_predictor_columns]
    )
    activity_test_scaled = scaler.transform(
        test_predictors[activity_predictor_columns]
    )

    full_training_matrix = np.hstack([
        training_predictors[covariate_columns].values,
        activity_training_scaled,
    ])
    full_test_matrix = np.hstack([
        test_predictors[covariate_columns].values,
        activity_test_scaled,
    ])

    # Alpha selection via RidgeCV on the training set
    ridge_model = RidgeCV(
        alphas=np.logspace(-3, 3, 25),
        cv=cross_validation_folds,
    )
    ridge_model.fit(full_training_matrix, training_outcome)

    # Cross-validated R² on the training set using the selected alpha
    cross_validation_scores = cross_val_score(
        estimator=RidgeCV(
            alphas=np.logspace(-3, 3, 25),
            cv=cross_validation_folds,
        ),
        X=full_training_matrix,
        y=training_outcome,
        cv=KFold(
            n_splits=cross_validation_folds,
            shuffle=True,
            random_state=random_state,
        ),
        scoring="r2",
    )

    test_predictions = ridge_model.predict(full_test_matrix)
    test_r2 = r2_score(test_outcome, test_predictions)
    test_rmse = float(np.sqrt(mean_squared_error(test_outcome, test_predictions)))
    test_mae = float(mean_absolute_error(test_outcome, test_predictions))

    print(
        f"\n  Cross-validated R² (train, 10-fold) : "
        f"{cross_validation_scores.mean():.4f}\n"
        f"  Test R²                             : {test_r2:.4f}\n"
        f"  Test RMSE                           : {test_rmse:.4f}\n"
        f"  Test MAE                            : {test_mae:.4f}\n"
        f"  Optimal alpha                       : {ridge_model.alpha_:.6f}"
    )

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "cross_validated_r2": float(cross_validation_scores.mean()),
        "test_r2": test_r2,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "optimal_alpha": float(ridge_model.alpha_),
    }


#### Execute Ridge for all outcomes

In [26]:
ridge_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_ridge_for_outcome(
                outcome_column=outcome_column,
                dataframe=modelling_dataframe,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                structural_covariate_column=STRUCTURAL_COVARIATE_COLUMN,
                include_structural_covariate=True,
                cross_validation_folds=CROSS_VALIDATION_FOLDS,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    ridge_results[group_name] = group_results


Group: pain  (5 outcomes)

  Outcome: koos_pain_symptomatic_side
  --------------------------------------------------
  Complete cases : 1,736 of 1,845 (94.1 %)
  Training set   : 1,388 participants
  Test set       : 348 participants

  Cross-validated R² (train, 10-fold) : 0.1301
  Test R²                             : 0.1368
  Test RMSE                           : 16.2789
  Test MAE                            : 12.9739
  Optimal alpha                       : 562.341325

  Outcome: icoap_total_symptomatic_side
  --------------------------------------------------
  Complete cases : 1,733 of 1,845 (93.9 %)
  Training set   : 1,386 participants
  Test set       : 347 participants

  Cross-validated R² (train, 10-fold) : 0.0798
  Test R²                             : 0.1308
  Test RMSE                           : 12.8644
  Test MAE                            : 9.1322
  Optimal alpha                       : 316.227766

  Outcome: pain_7day_symptomatic_side
  -----------------------------

#### Ridge summary table

In [27]:
summary_rows = []

for group_name, group_results in ridge_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "cv_r2_train": round(result["cross_validated_r2"], 4),
            "test_r2": round(result["test_r2"], 4),
            "test_rmse": round(result["test_rmse"], 4),
            "test_mae": round(result["test_mae"], 4),
            "optimal_alpha": round(result["optimal_alpha"], 6),
        })

ridge_summary_table = pd.DataFrame(summary_rows)

print("\nStage 2 — Ridge summary (sorted by test R²):")
print(
    ridge_summary_table
    .sort_values("test_r2", ascending=False)
    .to_string(index=False)
)


Stage 2 — Ridge summary (sorted by test R²):
                 group                        outcome  n_training  cv_r2_train  test_r2  test_rmse  test_mae  optimal_alpha
              function                     V0620MPACE        1364       0.2269   0.3361     0.1814    0.1424     316.227766
              function                     V06STEPST1        1364       0.2755   0.3350     3.5847    2.6960     177.827941
              function                     V06400MTIM        1248       0.2345   0.2117    49.8131   33.9983     316.227766
self_reported_function        dirkn1_symptomatic_side        1384       0.0888   0.1712     0.8247    0.6729     316.227766
self_reported_function        dirkn5_symptomatic_side        1341       0.0881   0.1621     0.9650    0.7967     316.227766
self_reported_function        dirkn2_symptomatic_side        1383       0.1073   0.1469     0.8664    0.7112     316.227766
self_reported_function       dirkn13_symptomatic_side        1149       0.0676   0.139

### 2.2.3. Elastic Net

## Purpose

Elastic Net is the second predictive model in the pipeline and serves
as the bridge between LASSO (Stage 1) and Ridge (Stage 2). It combines
L1 and L2 regularisation in a single penalty term controlled by two
hyperparameters: alpha (overall regularisation strength) and l1_ratio
(the balance between L1 and L2).

## Why Elastic Net after LASSO and Ridge

LASSO produces sparse solutions but struggles when predictors are
correlated — it tends to arbitrarily select one from a correlated
group and zero out the rest. Ridge retains all predictors but cannot
perform selection. Elastic Net inherits the selection property of
LASSO while using the L2 component to stabilise coefficient estimates
across correlated predictors. Given the moderate intercorrelations
among bout-structure and harmonic features in this predictor set,
Elastic Net may outperform both pure L1 and pure L2 models for some
outcomes.

## Hyperparameter tuning

Both alpha and l1_ratio are tuned jointly via ElasticNetCV using
10-fold cross-validation on the training set. The l1_ratio grid
covers the full spectrum from predominantly Ridge (0.1) to pure
LASSO (1.0), with denser coverage in the sparser region:
[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]. This allows the model to
select pure LASSO behaviour when sparsity is optimal, or a blend
when correlated predictors benefit from joint shrinkage.

Activity predictors are standardised on the training set; the same
scaler is applied to the test set. Covariates are passed through
unscaled, consistent with Stages 1 and 2.

## Output

elastic_net_results: dict mapping outcome_group_name -> list of
per-outcome result dicts. Each result dict contains:
- outcome_column
- n_training, n_test
- cross_validated_r2 (training set, 10-fold CV)
- test_r2
- test_rmse
- test_mae
- optimal_alpha
- optimal_l1_ratio

In [28]:
def run_elastic_net_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    structural_covariate_column: str,
    include_structural_covariate: bool = True,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit an Elastic Net model with jointly tuned alpha and l1_ratio for
    a single outcome and return held-out test set metrics.

    Alpha and l1_ratio are selected jointly via ElasticNetCV using
    10-fold cross-validation on the training set. The l1_ratio grid
    covers [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0], allowing the model
    to select anywhere from predominantly Ridge to pure LASSO behaviour.

    Activity predictors are standardised on the training set; the same
    scaler is applied to the test set without refitting. Covariates are
    passed through unscaled, consistent with Stages 1 and 2.

    A cross-validated R² on the training set is computed after fitting
    to provide a training-set estimate directly comparable across stages.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names
        (standardised before fitting).
    :param demographic_covariate_columns: Demographic covariate column
        names (passed through unscaled).
    :param structural_covariate_column: KL grade column name.
    :param include_structural_covariate: Whether to include KL grade as
        a covariate. Always True except for Stage 6.
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys outcome_column, n_training, n_test,
        cross_validated_r2, test_r2, test_rmse, test_mae,
        optimal_alpha, optimal_l1_ratio.
    """
    covariate_columns = list(demographic_covariate_columns)
    if include_structural_covariate:
        covariate_columns.append(structural_covariate_column)

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        structural_covariate_column=structural_covariate_column,
        include_structural_covariate=include_structural_covariate,
    )

    scaler = StandardScaler()
    activity_training_scaled = scaler.fit_transform(
        training_predictors[activity_predictor_columns]
    )
    activity_test_scaled = scaler.transform(
        test_predictors[activity_predictor_columns]
    )

    full_training_matrix = np.hstack([
        training_predictors[covariate_columns].values,
        activity_training_scaled,
    ])
    full_test_matrix = np.hstack([
        test_predictors[covariate_columns].values,
        activity_test_scaled,
    ])

    elastic_net_model = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
        cv=cross_validation_folds,
        random_state=random_state,
        max_iter=50_000,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        elastic_net_model.fit(full_training_matrix, training_outcome)

    cross_validation_scores = cross_val_score(
        estimator=ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
            cv=cross_validation_folds,
            random_state=random_state,
            max_iter=50_000,
        ),
        X=full_training_matrix,
        y=training_outcome,
        cv=KFold(
            n_splits=cross_validation_folds,
            shuffle=True,
            random_state=random_state,
        ),
        scoring="r2",
    )

    test_predictions = elastic_net_model.predict(full_test_matrix)
    test_r2 = r2_score(test_outcome, test_predictions)
    test_rmse = float(np.sqrt(mean_squared_error(test_outcome, test_predictions)))
    test_mae = float(mean_absolute_error(test_outcome, test_predictions))

    print(
        f"\n  Cross-validated R² (train, 10-fold) : "
        f"{cross_validation_scores.mean():.4f}\n"
        f"  Test R²                             : {test_r2:.4f}\n"
        f"  Test RMSE                           : {test_rmse:.4f}\n"
        f"  Test MAE                            : {test_mae:.4f}\n"
        f"  Optimal alpha                       : "
        f"{elastic_net_model.alpha_:.6f}\n"
        f"  Optimal l1_ratio                    : "
        f"{elastic_net_model.l1_ratio_:.2f}"
    )

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "cross_validated_r2": float(cross_validation_scores.mean()),
        "test_r2": test_r2,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "optimal_alpha": float(elastic_net_model.alpha_),
        "optimal_l1_ratio": float(elastic_net_model.l1_ratio_),
    }

#### Execute Elastic Net for all outcomes

In [29]:
elastic_net_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_elastic_net_for_outcome(
                outcome_column=outcome_column,
                dataframe=modelling_dataframe,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                structural_covariate_column=STRUCTURAL_COVARIATE_COLUMN,
                include_structural_covariate=True,
                cross_validation_folds=CROSS_VALIDATION_FOLDS,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    elastic_net_results[group_name] = group_results



Group: pain  (5 outcomes)

  Outcome: koos_pain_symptomatic_side
  --------------------------------------------------
  Complete cases : 1,736 of 1,845 (94.1 %)
  Training set   : 1,388 participants
  Test set       : 348 participants

  Cross-validated R² (train, 10-fold) : 0.1355
  Test R²                             : 0.1426
  Test RMSE                           : 16.2236
  Test MAE                            : 12.9090
  Optimal alpha                       : 0.336356
  Optimal l1_ratio                    : 1.00

  Outcome: icoap_total_symptomatic_side
  --------------------------------------------------
  Complete cases : 1,733 of 1,845 (93.9 %)
  Training set   : 1,386 participants
  Test set       : 347 participants

  Cross-validated R² (train, 10-fold) : 0.0782
  Test R²                             : 0.1330
  Test RMSE                           : 12.8485
  Test MAE                            : 9.0903
  Optimal alpha                       : 0.259137
  Optimal l1_ratio           

#### Elastic Net summary table

In [30]:
summary_rows = []

for group_name, group_results in elastic_net_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "cv_r2_train": round(result["cross_validated_r2"], 4),
            "test_r2": round(result["test_r2"], 4),
            "test_rmse": round(result["test_rmse"], 4),
            "test_mae": round(result["test_mae"], 4),
            "optimal_alpha": round(result["optimal_alpha"], 6),
            "optimal_l1_ratio": round(result["optimal_l1_ratio"], 2),
        })

elastic_net_summary_table = pd.DataFrame(summary_rows)

print("\nStage 3 — Elastic Net summary (sorted by test R²):")
print(
    elastic_net_summary_table
    .sort_values("test_r2", ascending=False)
    .to_string(index=False)
)


Stage 3 — Elastic Net summary (sorted by test R²):
                 group                        outcome  n_training  cv_r2_train  test_r2  test_rmse  test_mae  optimal_alpha  optimal_l1_ratio
              function                     V0620MPACE        1364       0.2248   0.3400     0.1809    0.1419       0.029585               0.1
              function                     V06STEPST1        1364       0.2769   0.3354     3.5838    2.6980       0.080257               0.3
              function                     V06400MTIM        1248       0.2339   0.2133    49.7633   33.9261       0.629154               0.9
self_reported_function        dirkn1_symptomatic_side        1384       0.0943   0.1687     0.8260    0.6728       0.013982               1.0
self_reported_function        dirkn5_symptomatic_side        1341       0.0893   0.1682     0.9615    0.7909       0.010254               1.0
self_reported_function        dirkn2_symptomatic_side        1383       0.1111   0.1521     0.86

### 2.2.4. Random Forest

## Purpose

Random Forest is the first non-linear model in the pipeline. Unlike
the three linear models in Stages 1–3, Random Forest makes no
assumption about the functional form of the relationship between
predictors and outcomes. It can capture interactions between
predictors and non-linear effects that linear regularisation cannot
represent, making it a useful benchmark for whether the activity-
outcome relationships in this dataset are adequately described by
linear models.

## Why Random Forest at this stage

If Random Forest substantially outperforms the linear models on the
held-out test set, this indicates that non-linear or interaction
effects are present and that the linear models are underfitting. If
performance is comparable, the simpler linear models are sufficient
and preferable for interpretability. This comparison is one of the
central methodological questions Stage 7 will answer.

## Model specification

A single Random Forest regressor is fitted per outcome with the
following fixed hyperparameters:

- n_estimators = 500: enough trees for stable importance estimates
  at this sample size without excessive computation time
- max_depth = None: trees grow until leaves are pure or contain
  fewer than min_samples_leaf samples, allowing the model to
  capture deep interactions if they exist
- min_samples_leaf = 10: minimum 10 participants per leaf node,
  chosen to prevent overfitting on a dataset of approximately
  1,300–1,400 training participants
- No standardisation is applied: tree-based models are scale-
  invariant and do not require feature scaling

Hyperparameters are not grid-searched here. The fixed specification
is a deliberate choice for a dataset of this size — grid search over
Random Forest hyperparameters is computationally expensive and the
chosen defaults are well-validated for n ≈ 1,000–2,000 in the
literature.

## Feature importance

Mean decrease in impurity (MDI) importance is reported for the top
10 predictors per outcome. M

In [31]:
def run_random_forest_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    structural_covariate_column: str,
    include_structural_covariate: bool = True,
    number_of_trees: int = 500,
    minimum_samples_per_leaf: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit a Random Forest regressor for a single outcome and return
    held-out test set metrics and feature importances.

    No standardisation is applied since tree-based models are scale-
    invariant. Hyperparameters are fixed rather than grid-searched:
    500 trees and a minimum of 10 samples per leaf are well-validated
    defaults for datasets of approximately 1,000–2,000 participants.

    Feature importances are mean decrease in impurity (MDI), reported
    for all predictors ranked by importance descending. MDI is biased
    toward high-cardinality continuous predictors; rankings should be
    read as indicative rather than definitive.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names.
    :param structural_covariate_column: KL grade column name.
    :param include_structural_covariate: Whether to include KL grade as
        a covariate. Always True except for Stage 6.
    :param number_of_trees: Number of trees in the forest (default 500).
    :param minimum_samples_per_leaf: Minimum number of participants per
        leaf node (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys outcome_column, n_training, n_test,
        test_r2, test_rmse, test_mae, feature_importances.
    """
    covariate_columns = list(demographic_covariate_columns)
    if include_structural_covariate:
        covariate_columns.append(structural_covariate_column)

    all_predictor_columns = covariate_columns + activity_predictor_columns

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        structural_covariate_column=structural_covariate_column,
        include_structural_covariate=include_structural_covariate,
    )

    training_matrix = training_predictors[all_predictor_columns].values
    test_matrix = test_predictors[all_predictor_columns].values

    random_forest_model = RandomForestRegressor(
        n_estimators=number_of_trees,
        max_depth=None,
        min_samples_leaf=minimum_samples_per_leaf,
        n_jobs=-1,
        random_state=random_state,
    )
    random_forest_model.fit(training_matrix, training_outcome)

    test_predictions = random_forest_model.predict(test_matrix)
    test_r2 = r2_score(test_outcome, test_predictions)
    test_rmse = float(np.sqrt(mean_squared_error(test_outcome, test_predictions)))
    test_mae = float(mean_absolute_error(test_outcome, test_predictions))

    feature_importances = pd.Series(
        data=random_forest_model.feature_importances_,
        index=all_predictor_columns,
        name="importance",
    ).sort_values(ascending=False)

    print(
        f"\n  Test R²   : {test_r2:.4f}\n"
        f"  Test RMSE : {test_rmse:.4f}\n"
        f"  Test MAE  : {test_mae:.4f}\n"
        f"\n  Top 10 feature importances (MDI):"
    )
    for predictor_name, importance_value in feature_importances.head(10).items():
        print(f"    {predictor_name:<45} {importance_value:.4f}")

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "test_r2": test_r2,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "feature_importances": feature_importances,
    }

#### Execute Random Forest for all outcomes

In [32]:
random_forest_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_random_forest_for_outcome(
                outcome_column=outcome_column,
                dataframe=modelling_dataframe,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                structural_covariate_column=STRUCTURAL_COVARIATE_COLUMN,
                include_structural_covariate=True,
                number_of_trees=500,
                minimum_samples_per_leaf=10,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    random_forest_results[group_name] = group_results



Group: pain  (5 outcomes)

  Outcome: koos_pain_symptomatic_side
  --------------------------------------------------
  Complete cases : 1,736 of 1,845 (94.1 %)
  Training set   : 1,388 participants
  Test set       : 348 participants

  Test R²   : 0.1078
  Test RMSE : 16.5498
  Test MAE  : 12.9491

  Top 10 feature importances (MDI):
    kl_grade_index_knee                           0.1619
    V06BMI                                        0.0951
    wake_minute_sd                                0.0619
    mean_sedentary_bout_total_minutes             0.0534
    V06AGE                                        0.0415
    mean_mvpa_bout_mean_duration                  0.0357
    mean_mvpa_bout_count                          0.0335
    mean_sedentary_bout_count                     0.0318
    iv_contrast                                   0.0318
    interdaily_stability                          0.0314

  Outcome: icoap_total_symptomatic_side
  ------------------------------------------------

#### Random Forest summary table

In [33]:
summary_rows = []

for group_name, group_results in random_forest_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "test_r2": round(result["test_r2"], 4),
            "test_rmse": round(result["test_rmse"], 4),
            "test_mae": round(result["test_mae"], 4),
        })

random_forest_summary_table = pd.DataFrame(summary_rows)

print("\nStage 4 — Random Forest summary (sorted by test R²):")
print(
    random_forest_summary_table
    .sort_values("test_r2", ascending=False)
    .to_string(index=False)
)


Stage 4 — Random Forest summary (sorted by test R²):
                 group                        outcome  n_training  test_r2  test_rmse  test_mae
              function                     V06STEPST1        1364   0.3762     3.4720    2.5892
              function                     V0620MPACE        1364   0.3468     0.1799    0.1403
              function                     V06400MTIM        1248   0.2465    48.7012   33.3090
self_reported_function        dirkn1_symptomatic_side        1384   0.1357     0.8422    0.6982
                   adl                       V06LLD6A        1386   0.1330     1.0966    0.8992
self_reported_function        dirkn8_symptomatic_side        1384   0.1197     0.7677    0.5542
self_reported_function        dirkn2_symptomatic_side        1383   0.1193     0.8803    0.7246
                  pain   icoap_total_symptomatic_side        1386   0.1080    13.0324    9.2384
                  pain     koos_pain_symptomatic_side        1388   0.1078    16.5

### 2.2.5. XGBoost

## Purpose

XGBoost is the final and most flexible model in the pipeline. It fits
an ensemble of gradient-boosted decision trees sequentially, where
each tree corrects the residual errors of the previous ensemble.
Compared to Random Forest (Stage 4), which builds trees independently
in parallel, XGBoost uses a more targeted learning process that often
achieves higher predictive accuracy at the cost of additional
hyperparameters requiring tuning.

## Why XGBoost after Random Forest

Random Forest established whether non-linear effects are present.
XGBoost answers whether those non-linear effects can be exploited
more efficiently with a sequential boosting strategy. If XGBoost
substantially outperforms Random Forest, the outcome-predictor
relationships contain structured residual patterns that boosting
can exploit. If performance is comparable, Random Forest's simpler
parallel averaging is sufficient.

## Hyperparameter tuning

A focused grid search is performed over two hyperparameters:

- max_depth: [3, 4, 6] — controls tree complexity and the depth
  of interactions captured. Shallow trees (3) produce smoother,
  more regularised fits; deeper trees (6) can capture higher-order
  interactions at the risk of overfitting.
- learning_rate: [0.03, 0.05, 0.1] — controls the contribution
  of each tree to the ensemble. Lower rates require more trees but
  generalise better.

All other hyperparameters are fixed:
- n_estimators = 500: sufficient trees for convergence at the
  learning rates used
- subsample = 0.8: each tree is fitted on 80% of the training
  data, reducing variance
- colsample_bytree = 0.8: each tree uses 80% of predictors,
  further reducing variance and computation time

Grid search uses 10-fold cross-validation on the training set with
R² as the scoring metric, yielding 9 configurations × 10 folds =
90 fits per outcome. No standardisation is applied since tree-based
models are scale-invariant.

## Feature importance

Gain-based feature importance is reported for the top 10 predictors
per outcome. Gain measures the average improvement in the loss
function brought by a feature across all splits where it is used,
making it more informative than the frequency-based importance used
in Random Forest.

## Output

xgboost_results: dict mapping outcome_group_name -> list of
per-outcome result dicts. Each result dict contains:
- outcome_column
- n_training, n_test
- cross_validated_r2 (training set, best CV score from grid search)
- test_r2
- test_rmse
- test_mae
- best_hyperparameters
- feature_importances (Series, all predictors ranked by gain)

In [34]:
def run_xgboost_for_outcome(
    *,
    outcome_column: str,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    structural_covariate_column: str,
    include_structural_covariate: bool = True,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit an XGBoost gradient-boosted regressor with grid-searched
    hyperparameters for a single outcome and return held-out test
    set metrics and feature importances.

    A focused grid search over max_depth [3, 4, 6] and learning_rate
    [0.03, 0.05, 0.1] is performed using 10-fold cross-validation on
    the training set. All other hyperparameters are fixed: 500 trees,
    subsample 0.8, colsample_bytree 0.8.

    No standardisation is applied since tree-based models are scale-
    invariant. Feature importances are gain-based, measuring the
    average loss improvement per split where each feature is used.

    :param outcome_column: Name of the outcome variable to model.
    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names.
    :param structural_covariate_column: KL grade column name.
    :param include_structural_covariate: Whether to include KL grade as
        a covariate. Always True except for Stage 6.
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys outcome_column, n_training, n_test,
        cross_validated_r2, test_r2, test_rmse, test_mae,
        best_hyperparameters, feature_importances.
    """
    covariate_columns = list(demographic_covariate_columns)
    if include_structural_covariate:
        covariate_columns.append(structural_covariate_column)

    all_predictor_columns = covariate_columns + activity_predictor_columns

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=outcome_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        structural_covariate_column=structural_covariate_column,
        include_structural_covariate=include_structural_covariate,
    )

    training_matrix = training_predictors[all_predictor_columns].values
    test_matrix = test_predictors[all_predictor_columns].values

    hyperparameter_grid = {
        "max_depth": [3, 4, 6],
        "learning_rate": [0.03, 0.05, 0.1],
    }

    grid_search = GridSearchCV(
        estimator=xgboost.XGBRegressor(
            n_estimators=500,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        ),
        param_grid=hyperparameter_grid,
        cv=KFold(
            n_splits=cross_validation_folds,
            shuffle=True,
            random_state=random_state,
        ),
        scoring="r2",
        n_jobs=-1,
    )
    grid_search.fit(training_matrix, training_outcome)

    best_model = grid_search.best_estimator_
    test_predictions = best_model.predict(test_matrix)
    test_r2 = r2_score(test_outcome, test_predictions)
    test_rmse = float(np.sqrt(mean_squared_error(test_outcome, test_predictions)))
    test_mae = float(mean_absolute_error(test_outcome, test_predictions))

    feature_importances = pd.Series(
        data=best_model.feature_importances_,
        index=all_predictor_columns,
        name="importance",
    ).sort_values(ascending=False)

    print(
        f"\n  Cross-validated R² (train, 10-fold) : "
        f"{grid_search.best_score_:.4f}\n"
        f"  Test R²                             : {test_r2:.4f}\n"
        f"  Test RMSE                           : {test_rmse:.4f}\n"
        f"  Test MAE                            : {test_mae:.4f}\n"
        f"  Best hyperparameters                : "
        f"{grid_search.best_params_}\n"
        f"\n  Top 10 feature importances (gain):"
    )
    for predictor_name, importance_value in feature_importances.head(10).items():
        print(f"    {predictor_name:<45} {importance_value:.4f}")

    return {
        "outcome_column": outcome_column,
        "n_training": len(training_predictors),
        "n_test": len(test_predictors),
        "cross_validated_r2": float(grid_search.best_score_),
        "test_r2": test_r2,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "best_hyperparameters": grid_search.best_params_,
        "feature_importances": feature_importances,
    }

#### Execute XGBoost for all outcomes

In [35]:
xgboost_results: dict[str, list[dict]] = {}

for group_name, outcome_columns in OUTCOME_GROUPS.items():
    print(f"\n{'=' * 70}")
    print(f"Group: {group_name}  ({len(outcome_columns)} outcomes)")
    print(f"{'=' * 70}")

    group_results = []

    for outcome_column in outcome_columns:
        print(f"\n  Outcome: {outcome_column}")
        print(f"  {'-' * 50}")

        try:
            result = run_xgboost_for_outcome(
                outcome_column=outcome_column,
                dataframe=modelling_dataframe,
                activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
                demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
                structural_covariate_column=STRUCTURAL_COVARIATE_COLUMN,
                include_structural_covariate=True,
                cross_validation_folds=CROSS_VALIDATION_FOLDS,
                random_state=RANDOM_STATE,
            )
            group_results.append(result)

        except ValueError as error:
            print(f"  Skipped: {error}")

    xgboost_results[group_name] = group_results


Group: pain  (5 outcomes)

  Outcome: koos_pain_symptomatic_side
  --------------------------------------------------
  Complete cases : 1,736 of 1,845 (94.1 %)
  Training set   : 1,388 participants
  Test set       : 348 participants

  Cross-validated R² (train, 10-fold) : 0.0643
  Test R²                             : 0.1061
  Test RMSE                           : 16.5655
  Test MAE                            : 12.8742
  Best hyperparameters                : {'learning_rate': 0.03, 'max_depth': 3}

  Top 10 feature importances (gain):
    kl_grade_index_knee                           0.0987
    V06COMORB                                     0.0499
    V06BMI                                        0.0431
    mean_mvpa_bout_count                          0.0426
    wake_minute_sd                                0.0414
    mean_sedentary_bout_total_minutes             0.0409
    mean_mvpa_bout_mean_duration                  0.0404
    V06AGE                                        0.0369

#### XGBoost summary table

In [36]:
summary_rows = []

for group_name, group_results in xgboost_results.items():
    for result in group_results:
        summary_rows.append({
            "group": group_name,
            "outcome": result["outcome_column"],
            "n_training": result["n_training"],
            "cv_r2_train": round(result["cross_validated_r2"], 4),
            "test_r2": round(result["test_r2"], 4),
            "test_rmse": round(result["test_rmse"], 4),
            "test_mae": round(result["test_mae"], 4),
            "max_depth": result["best_hyperparameters"]["max_depth"],
            "learning_rate": result["best_hyperparameters"]["learning_rate"],
        })

xgboost_summary_table = pd.DataFrame(summary_rows)

print("\nStage 5 — XGBoost summary (sorted by test R²):")
print(
    xgboost_summary_table
    .sort_values("test_r2", ascending=False)
    .to_string(index=False)
)


Stage 5 — XGBoost summary (sorted by test R²):
                 group                        outcome  n_training  cv_r2_train  test_r2  test_rmse  test_mae  max_depth  learning_rate
              function                     V06STEPST1        1364       0.3355   0.3879     3.4392    2.5823          3           0.03
              function                     V0620MPACE        1364       0.2405   0.3460     0.1800    0.1426          3           0.03
              function                     V06400MTIM        1248       0.1934   0.2197    49.5610   33.0613          3           0.03
                   adl                      V06LLD10B        1385       0.0291   0.1557     0.8590    0.7028          3           0.03
                  pain     koos_pain_symptomatic_side        1388       0.0643   0.1061    16.5655   12.8742          3           0.03
self_reported_function        dirkn1_symptomatic_side        1384       0.0279   0.1060     0.8565    0.6922          3           0.03
       

### 2.2.6 KL grade classification

## Purpose

Stage 6 treats KL grade as a multiclass ordinal outcome rather than
a covariate. The question being asked is different from Stages 1–5:
not "how well do activity features predict a clinical symptom outcome"
but "how much does the activity pattern alone reveal about the
structural severity of a participant's knee osteoarthritis".

This is methodologically distinct because KL grade is a radiographic
measure of structural joint damage that is only loosely coupled to
symptoms. A participant with KL grade 4 may report minimal pain if
they have adapted their behaviour; a participant with KL grade 1 may
report severe pain due to sensitisation. If activity features
nonetheless predict KL grade above chance, this suggests that
structural severity leaves a detectable signature in movement patterns
independent of subjective symptom reporting.

## Why ordinal classification rather than regression

KL grade takes integer values from 0 to 4 with an ordered but not
necessarily interval-scaled relationship. Treating it as a continuous
regression target would impose the assumption that the step from
KL 0 to KL 1 represents the same magnitude of change as KL 3 to KL 4,
which is not clinically justified. Ordinal classification preserves
the ordering without imposing equal spacing.

## Evaluation metric

Quadratic weighted kappa (QWK) is the primary metric. QWK penalises
predictions in proportion to how far they are from the true class —
a prediction of KL 4 when the true grade is KL 0 is penalised much
more heavily than a prediction of KL 2 when the true grade is KL 1.
This is the appropriate metric for an ordinal outcome where adjacent
misclassifications are less serious than distant ones.

Accuracy and mean absolute error on the integer-coded grade are
reported as secondary metrics for interpretability.

## Class imbalance

KL grade is not uniformly distributed — KL 2 and KL 3 are the most
common grades while KL 4 is rare. Class imbalance is handled via:
- Random Forest: balanced class weights, which up-weight minority
  classes during tree fitting
- XGBoost: sample weights inversely proportional to class frequency,
  applied during fitting

## Model specification

Two classifiers are fitted:

Random Forest classifier:
- 500 trees, balanced class weights, min 10 samples per leaf
- No standardisation required

XGBoost classifier:
- 400 trees, max_depth 4, learning_rate 0.05
- subsample 0.8, colsample_bytree 0.8
- Inverse frequency sample weights for class imbalance
- Hyperparameters are fixed rather than grid-searched to keep
  runtime manageable; the specification follows the same
  rationale as Stage 5

## Important note on covariates

KL grade must not appear in the covariate block when it is the
outcome. The structural covariate (kl_grade_index_knee) is therefore
excluded and the model uses demographic covariates only: age, BMI,
comorbidity count, and employment status.

## Output

kl_grade_results: dict with keys random_forest and xgboost, each
containing:
- test_kappa (quadratic weighted kappa, primary metric)
- test_accuracy
- test_mean_absolute_error_grade
- cross_validated_kappa (training set, stratified 10-fold CV)
- feature_importances (Series, all predictors ranked)
- confusion_matrix (DataFrame, rows = actual, columns = predicted)

In [37]:
def run_kl_grade_classification(
    *,
    dataframe: pd.DataFrame,
    activity_predictor_columns: list[str],
    demographic_covariate_columns: list[str],
    structural_covariate_column: str,
    model_name: str,
    cross_validation_folds: int = 10,
    random_state: int = 42,
) -> dict:
    """
    Fit a multiclass classifier for KL grade (0 to 4) and evaluate
    using metrics appropriate for an ordinal outcome.

    KL grade is excluded from the covariate block to prevent data
    leakage. The train/test split is stratified by KL grade itself.
    Class imbalance is handled via balanced class weights for Random
    Forest and normalised inverse frequency sample weights for XGBoost.

    Quadratic weighted kappa is the primary evaluation metric. It
    penalises predictions in proportion to the squared distance from
    the true class, which is appropriate for an ordinal outcome where
    distant misclassifications are more serious than adjacent ones.

    :param dataframe: Modelling dataframe with one row per participant.
    :param activity_predictor_columns: Activity predictor column names.
    :param demographic_covariate_columns: Demographic covariate column
        names. Must not contain KL grade.
    :param structural_covariate_column: KL grade column name. Used only
        to verify it is absent from covariate_columns and to prepare
        the dataset split — not included as a predictor.
    :param model_name: Either ``"random_forest"`` or ``"xgboost"``.
    :param cross_validation_folds: Number of CV folds (default 10).
    :param random_state: Random seed for reproducibility.
    :returns: Dictionary with keys cross_validated_kappa, test_kappa,
        test_accuracy, test_mean_absolute_error_grade,
        feature_importances, confusion_matrix.
    :raises ValueError: If ``model_name`` is not recognised or if KL
        grade appears in ``demographic_covariate_columns``.
    """
    if structural_covariate_column in demographic_covariate_columns:
        raise ValueError(
            f"KL grade column '{structural_covariate_column}' must not "
            f"appear in demographic_covariate_columns when modelling KL "
            f"grade as the outcome — this would cause data leakage."
        )

    if model_name not in {"random_forest", "xgboost"}:
        raise ValueError(
            f"Unrecognised model_name '{model_name}'. "
            f"Expected 'random_forest' or 'xgboost'."
        )

    all_predictor_columns = (
        list(demographic_covariate_columns) + activity_predictor_columns
    )

    (
        training_predictors,
        training_outcome,
        test_predictors,
        test_outcome,
    ) = prepare_modelling_dataset_for_stage(
        dataframe=dataframe,
        outcome_column=structural_covariate_column,
        activity_predictor_columns=activity_predictor_columns,
        demographic_covariate_columns=demographic_covariate_columns,
        structural_covariate_column=structural_covariate_column,
        include_structural_covariate=False,
    )

    training_matrix = training_predictors[all_predictor_columns].values
    test_matrix = test_predictors[all_predictor_columns].values

    training_classes = training_outcome.astype(int).values
    test_classes = test_outcome.astype(int).values

    if model_name == "random_forest":
        classifier = RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=10,
            class_weight="balanced",
            n_jobs=-1,
            random_state=random_state,
        )
        classifier.fit(training_matrix, training_classes)

    else:
        # Scale each sample's weight by the inverse of its class frequency,
        # normalised so the total weight equals the number of training
        # samples. This avoids the over-correction that occurs with raw
        # inverse counts and prevents the classifier collapsing to a
        # single predicted class.
        class_counts = np.bincount(training_classes)
        class_weights = len(training_classes) / (
            len(class_counts) * class_counts
        )
        sample_weights = class_weights[training_classes]

        classifier = xgboost.XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softmax",
            num_class=int(np.max(training_classes)) + 1,
            random_state=random_state,
            n_jobs=-1,
            verbosity=0,
        )
        classifier.fit(
            training_matrix,
            training_classes,
            sample_weight=sample_weights,
        )

    def score_quadratic_weighted_kappa(
        estimator,
        predictor_matrix,
        true_values,
    ) -> float:
        """
        Scorer function for cross_val_score computing quadratic weighted
        kappa between true and predicted KL grade classes.

        :param estimator: Fitted classifier with a predict method.
        :param predictor_matrix: Feature matrix to predict from.
        :param true_values: True integer-coded KL grade values.
        :returns: Quadratic weighted kappa score.
        """
        return cohen_kappa_score(
            y1=true_values,
            y2=estimator.predict(predictor_matrix),
            weights="quadratic",
        )

    cross_validation_kappa_scores = cross_val_score(
        estimator=classifier.__class__(**classifier.get_params()),
        X=training_matrix,
        y=training_classes,
        cv=StratifiedKFold(
            n_splits=cross_validation_folds,
            shuffle=True,
            random_state=random_state,
        ),
        scoring=score_quadratic_weighted_kappa,
        n_jobs=-1,
    )

    test_predictions = classifier.predict(test_matrix)

    test_kappa = cohen_kappa_score(
        y1=test_classes,
        y2=test_predictions,
        weights="quadratic",
    )
    test_accuracy = accuracy_score(test_classes, test_predictions)
    test_mean_absolute_error_grade = float(
        np.mean(np.abs(test_classes - test_predictions))
    )

    feature_importances = pd.Series(
        data=classifier.feature_importances_,
        index=all_predictor_columns,
        name="importance",
    ).sort_values(ascending=False)

    confusion_matrix = pd.crosstab(
        pd.Series(test_classes, name="actual"),
        pd.Series(test_predictions, name="predicted"),
    )

    print(
        f"\n  Cross-validated kappa (train, {cross_validation_folds}-fold) : "
        f"{cross_validation_kappa_scores.mean():.4f}\n"
        f"  Test quadratic weighted kappa          : {test_kappa:.4f}\n"
        f"  Test accuracy                          : {test_accuracy:.4f}\n"
        f"  Test MAE (grade)                       : "
        f"{test_mean_absolute_error_grade:.4f}\n"
        f"\n  Confusion matrix (rows: actual, columns: predicted):\n"
        f"{confusion_matrix}\n"
        f"\n  Top 10 feature importances:"
    )
    for predictor_name, importance_value in feature_importances.head(10).items():
        print(f"    {predictor_name:<45} {importance_value:.4f}")

    return {
        "cross_validated_kappa": float(cross_validation_kappa_scores.mean()),
        "test_kappa": test_kappa,
        "test_accuracy": test_accuracy,
        "test_mean_absolute_error_grade": test_mean_absolute_error_grade,
        "feature_importances": feature_importances,
        "confusion_matrix": confusion_matrix,
    }

#### Execute KL grade classification for both models

In [38]:
print(f"\n{'=' * 70}")
print("Stage 6 — KL grade classification")
print(f"{'=' * 70}")

kl_grade_results: dict[str, dict] = {}

for model_name in ["random_forest", "xgboost"]:
    print(f"\n  Model: {model_name}")
    print(f"  {'-' * 50}")

    kl_grade_results[model_name] = run_kl_grade_classification(
        dataframe=modelling_dataframe,
        activity_predictor_columns=FINAL_PREDICTOR_COLUMNS,
        demographic_covariate_columns=DEMOGRAPHIC_COVARIATE_COLUMNS,
        structural_covariate_column=STRUCTURAL_COVARIATE_COLUMN,
        model_name=model_name,
        cross_validation_folds=CROSS_VALIDATION_FOLDS,
        random_state=RANDOM_STATE,
    )


Stage 6 — KL grade classification

  Model: random_forest
  --------------------------------------------------
  Complete cases : 1,736 of 1,845 (94.1 %)
  Training set   : 1,388 participants
  Test set       : 348 participants

  Cross-validated kappa (train, 10-fold) : 0.2085
  Test quadratic weighted kappa          : 0.2556
  Test accuracy                          : 0.2874
  Test MAE (grade)                       : 1.2270

  Confusion matrix (rows: actual, columns: predicted):
predicted   0   1   2   3   4
actual                       
0          35   7  23  15   5
1          17   5  21   9   5
2          22   9  34  20  14
3          12  10  18  22  16
4           2   3  10  10   4

  Top 10 feature importances:
    V06BMI                                        0.0633
    mean_mvpa_bout_mean_duration                  0.0440
    mean_mvpa_bout_count                          0.0436
    wake_minute_sd                                0.0432
    mean_active_bout_mean_duration           

#### KL grade classification summary table

In [39]:
print(f"\n{'=' * 70}")
print("Stage 6 — KL grade classification summary")
print(f"{'=' * 70}")

for model_name, result in kl_grade_results.items():
    print(
        f"\n  {model_name.replace('_', ' ').title()}\n"
        f"    CV kappa (train)          : "
        f"{result['cross_validated_kappa']:.4f}\n"
        f"    Test kappa (QWK)          : {result['test_kappa']:.4f}\n"
        f"    Test accuracy             : {result['test_accuracy']:.4f}\n"
        f"    Test MAE (grade)          : "
        f"{result['test_mean_absolute_error_grade']:.4f}"
    )

best_model_name = max(
    kl_grade_results,
    key=lambda name: kl_grade_results[name]["test_kappa"],
)
print(
    f"\n  Best model: {best_model_name.replace('_', ' ').title()} "
    f"(test kappa = "
    f"{kl_grade_results[best_model_name]['test_kappa']:.4f})"
)


Stage 6 — KL grade classification summary

  Random Forest
    CV kappa (train)          : 0.2085
    Test kappa (QWK)          : 0.2556
    Test accuracy             : 0.2874
    Test MAE (grade)          : 1.2270

  Xgboost
    CV kappa (train)          : 0.1496
    Test kappa (QWK)          : 0.2368
    Test accuracy             : 0.2730
    Test MAE (grade)          : 1.1983

  Best model: Random Forest (test kappa = 0.2556)


## 2.3 Cross-model comparison

## Purpose

Stage 7 assembles the results from all previous stages into a single
comparison table per outcome, enabling direct comparison of all five
model families on a common held-out test set metric. This is the
primary results table for the thesis modelling chapter.

## What is being compared

For continuous outcomes (Stages 1–5), the comparison metric is test
R² — the proportion of variance in the held-out test set explained
by each model. All models were evaluated on the same 20% held-out
test set produced by the same stratified split, so test R² values
are directly comparable across model families.

Stage 1 LASSO contributes full_r2_cv rather than a test set R²
because its role is explanatory rather than predictive. It is included
in the table for reference but is not directly comparable to the
held-out test R² values from Stages 2–5. The incremental_r2 from
Stage 1 is reported as a separate column.

For KL grade (Stage 6), the comparison metric is quadratic weighted
kappa. This is reported in a separate table.

## Best model selection

The best model per outcome is selected as the model family with the
highest test R² among Stages 2–5. Stage 1 is excluded from best
model selection because it optimises for explanation rather than
prediction.

## What the table reveals

Outcomes where all models produce similar test R² indicate that the
predictor-outcome relationship is well-described by linear models and
tree-based models offer no advantage. Outcomes where tree-based
models substantially outperform linear models indicate non-linear or
interaction effects. Outcomes where all models produce low test R²
across the board indicate weak or absent signal from the activity
predictor set for that outcome.

In [40]:
def build_continuous_outcome_comparison_table(
    *,
    lasso_results: dict[str, list[dict]],
    ridge_results: dict[str, list[dict]],
    elastic_net_results: dict[str, list[dict]],
    random_forest_results: dict[str, list[dict]],
    xgboost_results: dict[str, list[dict]],
) -> pd.DataFrame:
    """
    Assemble a single comparison table across all five model families
    for every continuous outcome modelled in Stages 1–5.

    For each outcome the table contains:
    - Stage 1 LASSO: incremental_r2 and full_r2_cv (explanatory,
      training set CV — not directly comparable to test R²)
    - Stages 2–5: test_r2 on the held-out test set
    - best_model: model family with the highest test R² among
      Stages 2–5
    - best_test_r2: the corresponding test R²

    :param lasso_results: Output of Stage 1 execution cell.
    :param ridge_results: Output of Stage 2 execution cell.
    :param elastic_net_results: Output of Stage 3 execution cell.
    :param random_forest_results: Output of Stage 4 execution cell.
    :param xgboost_results: Output of Stage 5 execution cell.
    :returns: DataFrame with one row per outcome sorted by
        best_test_r2 descending.
    """
    lasso_lookup: dict[str, dict] = {
        result["outcome_column"]: result
        for group_results in lasso_results.values()
        for result in group_results
    }
    ridge_lookup: dict[str, dict] = {
        result["outcome_column"]: result
        for group_results in ridge_results.values()
        for result in group_results
    }
    elastic_net_lookup: dict[str, dict] = {
        result["outcome_column"]: result
        for group_results in elastic_net_results.values()
        for result in group_results
    }
    random_forest_lookup: dict[str, dict] = {
        result["outcome_column"]: result
        for group_results in random_forest_results.values()
        for result in group_results
    }
    xgboost_lookup: dict[str, dict] = {
        result["outcome_column"]: result
        for group_results in xgboost_results.values()
        for result in group_results
    }

    all_outcome_columns = list(ridge_lookup.keys())

    rows = []

    for outcome_column in all_outcome_columns:
        lasso_result = lasso_lookup.get(outcome_column, {})
        ridge_result = ridge_lookup.get(outcome_column, {})
        elastic_net_result = elastic_net_lookup.get(outcome_column, {})
        random_forest_result = random_forest_lookup.get(outcome_column, {})
        xgboost_result = xgboost_lookup.get(outcome_column, {})

        ridge_test_r2 = ridge_result.get("test_r2", np.nan)
        elastic_net_test_r2 = elastic_net_result.get("test_r2", np.nan)
        random_forest_test_r2 = random_forest_result.get("test_r2", np.nan)
        xgboost_test_r2 = xgboost_result.get("test_r2", np.nan)

        predictive_model_r2_values = {
            "ridge": ridge_test_r2,
            "elastic_net": elastic_net_test_r2,
            "random_forest": random_forest_test_r2,
            "xgboost": xgboost_test_r2,
        }

        best_model_name = max(
            predictive_model_r2_values,
            key=lambda name: (
                predictive_model_r2_values[name]
                if not np.isnan(predictive_model_r2_values[name])
                else -np.inf
            ),
        )
        best_test_r2 = predictive_model_r2_values[best_model_name]

        group_name = next(
            (
                group
                for group, group_results in ridge_results.items()
                for result in group_results
                if result["outcome_column"] == outcome_column
            ),
            "unknown",
        )

        rows.append({
            "group": group_name,
            "outcome": outcome_column,
            "n_training": ridge_result.get("n_training", np.nan),
            "lasso_incremental_r2": round(
                lasso_result.get("incremental_r2", np.nan), 4
            ),
            "lasso_full_r2_cv": round(
                lasso_result.get("full_r2_cv", np.nan), 4
            ),
            "ridge_test_r2": round(ridge_test_r2, 4),
            "elastic_net_test_r2": round(elastic_net_test_r2, 4),
            "random_forest_test_r2": round(random_forest_test_r2, 4),
            "xgboost_test_r2": round(xgboost_test_r2, 4),
            "best_model": best_model_name,
            "best_test_r2": round(best_test_r2, 4),
        })

    comparison_table = (
        pd.DataFrame(rows)
        .sort_values("best_test_r2", ascending=False)
        .reset_index(drop=True)
    )

    return comparison_table


def print_comparison_table_by_group(
    comparison_table: pd.DataFrame,
) -> None:
    """
    Print the comparison table grouped by outcome domain for
    readability in the notebook output.

    :param comparison_table: Output of
        build_continuous_outcome_comparison_table.
    """
    column_widths = {
        "outcome": 40,
        "n_training": 10,
        "lasso_incremental_r2": 14,
        "lasso_full_r2_cv": 14,
        "ridge_test_r2": 13,
        "elastic_net_test_r2": 16,
        "random_forest_test_r2": 18,
        "xgboost_test_r2": 13,
        "best_model": 14,
        "best_test_r2": 12,
    }

    header = (
        f"{'Outcome':<40} {'n_train':>10} {'LASSO_incR²':>14} "
        f"{'LASSO_cvR²':>14} {'Ridge_R²':>13} {'ElNet_R²':>16} "
        f"{'RF_R²':>18} {'XGB_R²':>13} {'Best':>14} {'BestR²':>12}"
    )
    separator = "-" * len(header)

    for group_name, group_data in comparison_table.groupby("group", sort=False):
        print(f"\n{'=' * len(header)}")
        print(f"Group: {group_name}")
        print(f"{'=' * len(header)}")
        print(header)
        print(separator)

        for _, row in group_data.iterrows():
            print(
                f"{row['outcome']:<40} "
                f"{row['n_training']:>10} "
                f"{row['lasso_incremental_r2']:>14.4f} "
                f"{row['lasso_full_r2_cv']:>14.4f} "
                f"{row['ridge_test_r2']:>13.4f} "
                f"{row['elastic_net_test_r2']:>16.4f} "
                f"{row['random_forest_test_r2']:>18.4f} "
                f"{row['xgboost_test_r2']:>13.4f} "
                f"{row['best_model']:>14} "
                f"{row['best_test_r2']:>12.4f}"
            )

### Execute comparison table construction and printing

In [41]:
comparison_table = build_continuous_outcome_comparison_table(
    lasso_results=lasso_results,
    ridge_results=ridge_results,
    elastic_net_results=elastic_net_results,
    random_forest_results=random_forest_results,
    xgboost_results=xgboost_results,
)

print_comparison_table_by_group(comparison_table=comparison_table)

# Save results

comparison_table.to_csv(
    output_path / "stage_7_model_comparison.csv",
    index=False,
)

print(f"\nComparison table saved to: stage_7_model_comparison.csv")
print(f"Total outcomes compared: {len(comparison_table)}")



Group: function
Outcome                                     n_train    LASSO_incR²     LASSO_cvR²      Ridge_R²         ElNet_R²              RF_R²        XGB_R²           Best       BestR²
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
V06STEPST1                                     1364         0.0916         0.2747        0.3350           0.3354             0.3762        0.3879        xgboost       0.3879
V0620MPACE                                     1364         0.0531         0.2258        0.3361           0.3400             0.3468        0.3460  random_forest       0.3468
V06400MTIM                                     1248         0.0501         0.2285        0.2117           0.2133             0.2465        0.2197  random_forest       0.2465
V06CSPACE                                      1310         0.0015         0.1195        0.0729           0.0731 

#### KL grade classification summary

In [42]:
print(f"\n{'=' * 70}")
print("KL grade classification (Stage 6)")
print(f"{'=' * 70}")
print(
    f"\n{'Model':<20} {'CV kappa':>12} {'Test kappa':>12} "
    f"{'Accuracy':>12} {'MAE grade':>12}"
)
print("-" * 70)

for model_name, result in kl_grade_results.items():
    print(
        f"{model_name.replace('_', ' ').title():<20} "
        f"{result['cross_validated_kappa']:>12.4f} "
        f"{result['test_kappa']:>12.4f} "
        f"{result['test_accuracy']:>12.4f} "
        f"{result['test_mean_absolute_error_grade']:>12.4f}"
    )


KL grade classification (Stage 6)

Model                    CV kappa   Test kappa     Accuracy    MAE grade
----------------------------------------------------------------------
Random Forest              0.2085       0.2556       0.2874       1.2270
Xgboost                    0.1496       0.2368       0.2730       1.1983
